# ROMS-COSiNE Narragansett Bay Cost Function

This notebook compares ROMS-COSiNE station output with observations from Narragansett Bay, calculates normalized model-data mismatch costs, and writes a documented NetCDF cost-summary file. The output preserves the variable names and dimension order used by the legacy tuning workflow so existing analysis scripts can continue to read it.

## Scientific definition

The cost formulation follows [Ward et al. (2010)](https://www.sciencedirect.com/science/article/pii/S0924796309003431). For target (m), this notebook calculates

$$
J_m = \frac{1}{N_m\sigma_m^2}
      \sum_{n=1}^{N_m}\left(a_{n,m}-\hat{a}_{n,m}\right)^2,
$$

where:

- $N_m$ is the number of finite model-observation pairs;
- $a_{n,m}$ is the modeled value;
- $\hat{a}_{n,m}$ is the corresponding observation; and
- $\sigma_m\$ is the configured normalization standard deviation.

This is equivalent to the Ward et al. weight $W_m=C_m/\sigma_m$ with $C_m=1$. As in the legacy notebook, the overall (1/M) factor is omitted. The reported total is therefore the sum of finite component costs, not their mean.

The cost components can be summed and reweighted to achieve the exact formulation from [Ward et al. (2010)](https://www.sciencedirect.com/science/article/pii/S0924796309003431)

$$
J = \frac{1}{M} \sum_{m=1}^{M} W_m^2 \frac{1}{N_m} \sum_{n=1}^{N_m} (a - \hat{a})_{n,m}^2
$$

$M$: number of data targets \
$N_m$: number of observations for each target \
$\hat{a}$: observed value of data target $m$ at location/time $n$ \
$a$: model equivalent of $\hat{a}$ \
$W_m$: weight function; $W_m = \frac{C_m}{\sigma_m}$

<div class="alert alert-warning">
<strong>Comparability warning:</strong> A component without any finite model-observation pairs remains <code>NaN</code> and is excluded from the total. Compare total costs only among runs that use the same year and observational coverage.
</div>

## Output contents

The final NetCDF file contains:

1. Observations and corresponding model results on the legacy cost-summary coordinates.
2. Component costs and metadata describing counts, squared-error sums, normalization weights, and calculation status.
3. The finite-only total cost and a record of included and excluded components.
4. Model parameter values, provenance, processing choices, and software versions.

## Automation contract

| Category | Behavior |
| :--- | :--- |
| Required inputs | Station NetCDF, observational files, station-position file, and parameter metadata |
| Optional input | Annual production file; a missing file produces <code>NaN</code> production results and cost |
| Output | Validated NetCDF written to <code>cost_path</code> |
| Missing required input | Raise an exception and exit with a failure |
| Existing output | Warn and atomically replace it when overwriting is enabled |
| User interaction | None; execution is deterministic from the configuration cell |

<div class="alert alert-info">
<strong>Batch execution:</strong> This notebook contains no prompts, widgets, or confirmation dialogs. The configuration cell is tagged <code>parameters</code> so a Bash workflow can execute it directly or override values with a notebook runner such as Papermill.
</div>

In [ ]:
# Scientific data processing
import numpy as np
import pandas as pd
import xarray as xr
import gsw
import PyCO2SYS as pyco2

# Standard-library utilities
import json
import re
import sys
import tempfile
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# Notebook display helper; calculations do not depend on rich display.
from IPython.display import display

# Table of contents

The notebook is designed to run once, from top to bottom. Links use explicit anchors so they remain stable if headings are reworded.

| Stage | Purpose |
| :--- | :--- |
| [1. Resolve and load the station file](#load-station-file) | Validate configuration and locate the run-specific inputs and output. |
| [2. Identify model stations](#station-id) | Map descriptive station names to station-file indices. |
| [3. Read run metadata](#metadata) | Extract the model creation date and provenance. |
| [4. Validate observational files](#obs-data-files) | Define and verify every required observational path. |
| [5. Load observations](#load-observations) | Read each source once and standardize columns, dates, and missing values. |
| [6. Process and match model output](#process-station-file) | Convert time, calculate pH, aggregate results, and match observations. |
| &nbsp;&nbsp;[6.2. Convert UTC to Eastern time](#convert-model-time) | Preserve UTC values while creating local comparison timestamps. |
| &nbsp;&nbsp;[6.3. Calculate NBS-scale pH](#calculate-model-ph) | Apply TEOS-10 conversions and PyCO2SYS. |
| &nbsp;&nbsp;[6.9. Validate match coverage](#match-coverage) | Audit finite model-observation matches before constructing costs. |
| [7. Construct the cost-summary dataset](#construct-cost-summary) | Build and validate the legacy-compatible xarray schema. |
| &nbsp;&nbsp;[7.1. Define metadata](#metadata-definitions) | Record units, provenance, treatment, and citations. |
| &nbsp;&nbsp;[7.6. Check compatibility](#compatibility-checks) | Detect schema drift before cost calculation or saving. |
| [8. Calculate and save costs](#calculate-and-save-costs) | Calculate auditable costs and atomically write the final NetCDF file. |
| &nbsp;&nbsp;[8.1. Configure costs and output](#cost-settings) | Select weighting and overwrite behavior. |
| &nbsp;&nbsp;[8.3. Calculate costs](#calculate-costs) | Populate component costs, diagnostics, metadata, and total. |
| &nbsp;&nbsp;[8.4. Validate and save](#save-cost-summary) | Verify a temporary file before replacing <code>cost_path</code>. |

<a id="load-station-file"></a>
# 1. Resolve and load the station file

The run name and year identify the station input, production input, and final cost-summary path. Year-specific files are preferred when both generic and year-specific names exist.

In [ ]:
# -------------------------------------------------------------------------
# Batch parameters
# -------------------------------------------------------------------------
# Papermill overrides these values by inserting a new cell immediately after
# this one. Keep all validation and derived settings in the following setup
# cell so they are calculated from the injected values rather than defaults.
runname = "OPTUNA_74"
runyear = 2005

# Preserve historical weighting unless explicitly running a sensitivity test.
DEPTH_WEIGHT_MODE = "legacy"  # Allowed: "legacy", "per_depth"

# Batch-safe output controls; no interactive confirmation is requested.
SAVE_COST_FILE = True
OVERWRITE_EXISTING_COST_FILE = True

In [ ]:
# -------------------------------------------------------------------------
# Derived configuration
# -------------------------------------------------------------------------
# Filesystem roots are fixed for this workflow. All run-specific paths are
# resolved later from the injected runname and runyear values.
COST_DIR = Path("/Users/akbaskind/Desktop/COST_FILES")
DATA_DIR = Path("/Users/akbaskind/Desktop/Data")
OPT_DIR = Path("/Users/akbaskind/Desktop/Optimization")
SPOS_DIR = Path("/Users/akbaskind/Desktop/ROMS-Core-Docs")
FILE_LIST = OPT_DIR / "FileNames.xlsx"

# ROMS tuning runs span 2005-2024. Keeping the leap years explicit documents
# the calendar behavior expected by the model archive.
MODEL_YEARS = set(range(2005, 2025))
LEAP_YEARS = {2008, 2012, 2016, 2020, 2024}
VALID_DEPTH_WEIGHT_MODES = {"legacy", "per_depth"}

if not isinstance(runname, str) or not runname.strip():
    raise TypeError("runname must be a nonempty string.")
if isinstance(runyear, bool) or not isinstance(runyear, (int, np.integer)):
    raise TypeError("runyear must be an integer, not a string or Boolean.")
for option_name, option_value in {
    "SAVE_COST_FILE": SAVE_COST_FILE,
    "OVERWRITE_EXISTING_COST_FILE": OVERWRITE_EXISTING_COST_FILE,
}.items():
    if not isinstance(option_value, (bool, np.bool_)):
        raise TypeError(
            f"{option_name} must be a Boolean. Received "
            f"{option_value!r} ({type(option_value).__name__})."
        )

runname = runname.strip()
runyear = int(runyear)
if runyear not in MODEL_YEARS:
    raise ValueError(
        f"runyear must be between 2005 and 2024; received {runyear}."
    )
if DEPTH_WEIGHT_MODE not in VALID_DEPTH_WEIGHT_MODES:
    raise ValueError(
        "DEPTH_WEIGHT_MODE must be either 'legacy' or 'per_depth'; "
        f"received {DEPTH_WEIGHT_MODE!r}."
    )

is_leap_year = runyear in LEAP_YEARS
n_days = 366 if is_leap_year else 365
execution_started = time.perf_counter()


def installed_version(distribution_name):
    """Return a package version without making metadata optionality fatal."""
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return "unknown"


SOFTWARE_VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": installed_version("numpy"),
    "pandas": installed_version("pandas"),
    "xarray": installed_version("xarray"),
    "gsw": installed_version("gsw"),
    "PyCO2SYS": installed_version("PyCO2SYS"),
    "netCDF4": installed_version("netCDF4"),
}

In [ ]:
# ------------------------------------------------------------
# Resolve the run and its station file
# ------------------------------------------------------------
required_run_columns = ["Run Name", "Cost File", "Station File"]
runs = pd.read_excel(FILE_LIST)
runs = runs[required_run_columns].dropna(subset=required_run_columns)
run_names = runs["Run Name"].astype(str).str.strip()

requested_runname = str(runname).strip()
base_runname = "_".join(
    part for part in requested_runname.split("_")
    if part != str(runyear)
)
if not base_runname:
    raise ValueError(
        f"Could not derive a base run name from {requested_runname!r}."
    )

year_specific_runname = f"{base_runname}_{runyear}"
year_specific_station_path = (
    COST_DIR / f"ocean_sta_{year_specific_runname}.nc"
)
year_specific_cost_path = COST_DIR / f"{year_specific_runname}.nc"

# Prefer an explicitly year-specific run. Some valid historical files are
# present in COST_DIR even when FileNames.xlsx has not yet been updated, so
# the conventional filename is a documented fallback before the bare run.
if year_specific_runname in run_names.values:
    matched_runname = year_specific_runname
    run_match = runs.loc[run_names == matched_runname]
elif year_specific_station_path.is_file():
    matched_runname = year_specific_runname
    run_match = None
    station_path = year_specific_station_path
    cost_path = year_specific_cost_path
elif requested_runname in run_names.values:
    matched_runname = requested_runname
    run_match = runs.loc[run_names == matched_runname]
elif base_runname in run_names.values:
    matched_runname = base_runname
    run_match = runs.loc[run_names == matched_runname]
else:
    raise KeyError(
        "Could not find the requested run in FileNames.xlsx or COST_DIR.\n"
        f"Checked: {requested_runname!r}\n"
        f"Checked: {year_specific_runname!r}\n"
        f"Checked: {base_runname!r}"
    )

if run_match is not None:
    if len(run_match) > 1:
        raise ValueError(
            f"Found {len(run_match)} entries for {matched_runname!r}. "
            "The Run Name column should contain only one matching entry."
        )
    run_row = run_match.iloc[0]
    cost_path = COST_DIR / run_row["Cost File"]
    station_path = COST_DIR / run_row["Station File"]

# Production files always omit a duplicated year token from the run name.
production_runname = base_runname
PPPATH = COST_DIR / f"production_{runyear}_{production_runname}.csv"
production_file_missing = not PPPATH.is_file()

if not station_path.is_file():
    raise FileNotFoundError(f"Station file does not exist: {station_path}")

print("[1/8] Run files resolved")
print(f"  Requested run:   {requested_runname}")
print(f"  Matched entry:   {matched_runname}")
print(f"  Cost output:     {cost_path}")
print(f"  Station input:   {station_path}")
print(f"  Production input: {PPPATH}")
if production_file_missing:
    print("  WARNING: Production file missing; PP_mod will be filled with NaN.")

# STATIONS NEEDED BY THE COST FUNCTION
# ============================================================
key_station_names = {
    "PLT": "GSO phyto. series station (sta 2)",
    "GD": "GSO Dock",
    "PD": "PD (mean)",
    "BR": "BR (mean)",
    "CP": "CP (mean)",
    "NP": "NP (mean)",
    "MV": "MV (mean)",
    "QP": "QP (mean)",
    "PP": "PP (mean)",
    "TW": "TW (mean)",
    "GB": "GB (mean)",
    "MH": "MH (mean)",
    "SR": "SR (mean)",
    "CON": "Fulweiler Conimicut",
    "GI":  "Fulweiler Inner Greenwich Bay",
    "GM":  "Fulweiler Mid Greenwich Bay",
    "GO":  "Fulweiler Outer Greenwich Bay",
    **{
        f"O{i}": f"Oviatt production sta {i}"
        for i in range(1, 17)
    },
}

<a id="open-station-file"></a>
## 1.1. Open the station file

The root xarray dataset is named `sta`. A missing station file is fatal because no model-observation comparison can be performed without it.

In [ ]:
# Keep the source dataset open until the final file has been written. Most
# derived arrays are loaded or converted to DataFrames during processing.
sta = xr.open_dataset(station_path)
print(f"  Station dimensions: {dict(sta.sizes)}")

<a id="station-id"></a>
# 2. Identify model stations

Station indices are derived from the station-position file recorded in the station NetCDF metadata. Missing stations are retained later as all-`NaN` model records rather than changing the output schema.

In [ ]:
# The station NetCDF records the station-position input used by ROMS.
# Path.name prevents a machine-specific source path from leaking into the
# lookup performed against the local SPOS_DIR.
spos_name = Path(sta.attrs["spos_file"]).name
print(f"  Station-position input: {spos_name}")

<a id="station-index-function"></a>
## 2.1. Station-index lookup helper

In [ ]:
def get_station_index(station_input_path, sta_name):
    """
    Return the zero-based index of a named station in a ROMS station
    input file.

    The station name is taken from the comment following "!".
    """

    matches = []
    station_index = 0
    inside_station_list = False

    with open(station_input_path, "r") as file:

        for line_number, line in enumerate(file, start=1):

            # Station records begin after the POS header
            if line.strip().startswith("POS"):
                inside_station_list = True
                continue

            if not inside_station_list:
                continue

            # Skip blank lines and full-line comments
            stripped_line = line.strip()

            if not stripped_line or stripped_line.startswith("!"):
                continue

            # Station rows need a comment containing the station name
            if "!" not in line:
                continue

            station_values, station_comment = line.split("!", maxsplit=1)
            values = station_values.split()

            # A station row contains GRID, FLAG, longitude, latitude
            if len(values) < 4:
                continue

            try:
                float(values[2])
                float(values[3])
            except ValueError:
                continue

            # Normalize extra whitespace in the comment
            current_name = " ".join(station_comment.split())

            if current_name == sta_name:
                matches.append({
                    "index": station_index,
                    "line_number": line_number,
                    "name": current_name,
                })

            station_index += 1

    if not matches:
        raise KeyError(
            f"Station {sta_name!r} was not found in "
            f"{Path(station_input_path).name!r}."
        )

    if len(matches) > 1:
        raise ValueError(
            f"Station {sta_name!r} occurs {len(matches)} times in "
            f"{Path(station_input_path).name!r}."
        )

    return matches[0]["index"]

<a id="station-index-selection"></a>
## 2.2. Resolve configured station indices

In [ ]:
station_input_path = SPOS_DIR / spos_name
if not station_input_path.is_file():
    raise FileNotFoundError(
        f"Station input file does not exist: {station_input_path}"
    )

# Resolve each configured name exactly once. A missing station is recorded
# as None so downstream reindexing can preserve it as an all-NaN site.
station_indices = {}
missing_key_stations = {}
for abbreviation, station_name in key_station_names.items():
    try:
        station_indices[abbreviation] = get_station_index(
            station_input_path,
            station_name,
        )
    except KeyError:
        station_indices[abbreviation] = None
        missing_key_stations[abbreviation] = station_name

In [ ]:
available_station_indices = {
    name: index
    for name, index in station_indices.items()
    if index is not None
}

print("[2/8] Model stations resolved")
print(f"  Available configured stations: {len(available_station_indices)}")
print(f"  Missing configured stations:   {len(missing_key_stations)}")
for abbreviation, index in available_station_indices.items():
    print(
        f"    {abbreviation:<4} index={index:<3} "
        f"{key_station_names[abbreviation]}"
    )
if missing_key_stations:
    print(
        "  WARNING: Missing stations will be retained with NaN model values: "
        + ", ".join(missing_key_stations)
    )

In [ ]:
# Preserve the short names used by the scientific processing cells (PLT,
# GD, O1, etc.) while keeping station_indices as the authoritative mapping.
globals().update(station_indices)

<a id="metadata"></a>
# 3. Read run metadata

The run date is parsed from the station-file history attribute. If no parseable date is available, the value is recorded as `unknown` and a warning is printed.

In [ ]:
def get_run_date(ds, attribute_name="history"):
    """
    Extract the model run date from the station-file history attribute.

    Returns
    -------
    run_date : str
        Date formatted as M/D/YYYY, such as "1/18/2026".
    used_fallback : bool
        True when the fallback date was used.
    """

    def format_date(date):
        date = pd.to_datetime(date)
        return f"{date.month}/{date.day}/{date.year}"

    history = ds.attrs.get(attribute_name)

    if history is None:
        return "unknown", True

    date_match = re.search(
        r"\b(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December)\s+\d{1,2},\s+\d{4}\b",
        str(history),
    )

    if date_match is None:
        return "unknown", True

    run_date = pd.to_datetime(
        date_match.group(0),
        format="%B %d, %Y",
        errors="coerce",
    )

    if pd.isna(run_date):
        return "unknown", True

    return format_date(run_date), False

In [ ]:
rundate, run_date_used_fallback = get_run_date(sta)

print("[3/8] Run metadata read")
print(f"  Run date: {rundate}")
if run_date_used_fallback:
    print(
        "  WARNING: No parseable run date was found in station-file history."
    )

<a id="obs-data-files"></a>
# 4. Validate observational data files

<div class="alert alert-danger">
<strong>Required-input policy:</strong> Every observational file listed below must exist. The annual production file is handled separately and is the only optional observational/model diagnostic input.
</div>

In [ ]:
# Every observational filename is defined here so data provenance and
# required-file validation remain easy to audit.
WATER_QUALITY_DIR = DATA_DIR / "Narragansett Bay Water Quality"
NUTRIENT_DIR = DATA_DIR / "Narragansett Bay Nutrients"

OBS_FILES = {
    "nbfsmn_bottom": WATER_QUALITY_DIR / "NBFSMN_05_19_bottom.csv",
    "nbfsmn_surface": WATER_QUALITY_DIR / "NBFSMN_05_19_surface.csv",
    "plt_n_surface": NUTRIENT_DIR / "PLT_Nutrients_Surface_New.csv",
    "plt_n_bottom": NUTRIENT_DIR / "PLT_Nutrients_Bottom_New.csv",
    "chrp": NUTRIENT_DIR / "MERL_CHRP_Nutrients_Data_Processed.csv",
    "plt_si": NUTRIENT_DIR / "PLT_Nutrients_Si_New.csv",
    "plt_secchi": DATA_DIR / "PLT_Secchi.csv",
    "oviatt_pp": DATA_DIR / "OviattPPMeasurements.csv",
    "fulweiler_sediment": (
        DATA_DIR / "Fulweiler2007SedimentFluxData_TuningData.csv"
    ),
}

missing_obs_files = [
    path for path in OBS_FILES.values() if not path.is_file()
]
if missing_obs_files:
    missing_list = "\n".join(f"  - {path}" for path in missing_obs_files)
    raise FileNotFoundError(
        f"Missing observational data files:\n{missing_list}"
    )

print("[4/8] Observational files validated")
print(f"  Required files found: {len(OBS_FILES)}")

<a id="load-observations"></a>
# 5. Load and standardize observational data

Each source is read once. Only columns used by the cost function are loaded, common missing-value markers become `NaN`, dates are parsed consistently, and time-series sources are filtered to `runyear`.

In [ ]:
OBS_NA_VALUES = ["nd", "no data", ""]
NBFSMN_SITES = ["BR", "MV", "QP", "TW", "GB", "GD",
                "PD", "CP", "NP", "PP", "MH", "SR"]

def read_observation_csv(
    path,
    *,
    usecols,
    date_column=None,
    numeric_columns=(),
    replacements=None,
):
    """Read selected columns from one observational CSV and standardize types."""
    frame = pd.read_csv(
        path,
        usecols=usecols,
        na_values=OBS_NA_VALUES,
        low_memory=False,
    )

    if replacements:
        for column, values in replacements.items():
            frame[column] = frame[column].replace(values)

    if date_column is not None:
        frame[date_column] = pd.to_datetime(
            frame[date_column],
            errors="coerce",
            format="mixed",
        )

    if numeric_columns:
        frame[list(numeric_columns)] = frame[list(numeric_columns)].apply(
            pd.to_numeric,
            errors="coerce",
        )

    return frame


def select_year(frame, date_column="Date"):
    """Return observations from runyear with a clean, consecutive index."""
    return frame.loc[frame[date_column].dt.year.eq(runyear)].reset_index(drop=True)


def load_nbfsmn(path, *, depth, ph_column, oxygen_column):
    """Load one NBFSMN depth and give surface/bottom files one schema."""
    columns = ["date", "site", "S", "T", ph_column, oxygen_column, "Month", "Year"]
    frame = read_observation_csv(
        path,
        usecols=columns,
        date_column="date",
        numeric_columns=["S", "T", ph_column, oxygen_column, "Month", "Year"],
    ).rename(columns={ph_column: "pH", oxygen_column: "DO [mg/L]"})

    frame = frame.loc[
        frame["Year"].eq(runyear) & frame["site"].isin(NBFSMN_SITES)
    ].copy()
    frame["depth"] = depth
    frame["day_of_year"] = frame["date"].dt.dayofyear
    frame["DO [mmol/m3]"] = frame["DO [mg/L]"] * (1000 / 32)
    return frame.sort_values(["date", "site"]).reset_index(drop=True)

In [ ]:
# NBFSMN hydrography and water quality
nbfsmn_bottom = load_nbfsmn(
    OBS_FILES["nbfsmn_bottom"],
    depth="bottom",
    ph_column="pH [NBS]",
    oxygen_column="DO [mg/L]",
)
nbfsmn_surface = load_nbfsmn(
    OBS_FILES["nbfsmn_surface"],
    depth="surface",
    ph_column="pH",
    oxygen_column="DO",
)
nbfsmn_obs = pd.concat([nbfsmn_surface, nbfsmn_bottom], ignore_index=True)

# PLT nitrogen nutrients
plt_n_columns = ["Date", "NO3 (uM)", "NH4 (uM)"]
plt_n_surface = select_year(read_observation_csv(
    OBS_FILES["plt_n_surface"],
    usecols=plt_n_columns,
    date_column="Date",
    numeric_columns=plt_n_columns[1:],
)).assign(depth="surface")
plt_n_bottom = select_year(read_observation_csv(
    OBS_FILES["plt_n_bottom"],
    usecols=plt_n_columns,
    date_column="Date",
    numeric_columns=plt_n_columns[1:],
)).assign(depth="bottom")
plt_nutrients_obs = pd.concat([plt_n_surface, plt_n_bottom], ignore_index=True)

# CHRP nutrients
chrp_columns = [
    "Station", "Date", "NO3 [mmol m-3]", "NO2 [mmol m-3]",
    "NH3 [mmol m-3]", "Si [mmol m-3]", "Station Index[0]",
]
chrp_obs = select_year(read_observation_csv(
    OBS_FILES["chrp"],
    usecols=chrp_columns,
    date_column="Date",
    numeric_columns=chrp_columns[2:],
))

# PLT silicate and Secchi depth
plt_si_obs = select_year(read_observation_csv(
    OBS_FILES["plt_si"],
    usecols=["Date", "Surface Si", "Bottom Si"],
    date_column="Date",
    numeric_columns=["Surface Si", "Bottom Si"],
))
plt_secchi_obs = select_year(read_observation_csv(
    OBS_FILES["plt_secchi"],
    usecols=["Date", "Secchi Depth"],
    date_column="Date",
    numeric_columns=["Secchi Depth"],
    replacements={"Secchi Depth": {"4..2": 4.2}},
))

# Time-independent primary-production and sediment targets
oviatt_pp_obs = read_observation_csv(
    OBS_FILES["oviatt_pp"],
    usecols=["Station Number", "C-14 Production Estimate (gC m-2 y-1)"],
    numeric_columns=["Station Number", "C-14 Production Estimate (gC m-2 y-1)"],
).sort_values("Station Number").reset_index(drop=True)

sediment_columns = [
    "Date", "Site", "O2 [mmol m-2 d-1]",
    "NH4 [mmol m-2 d-1]", "NO3 [mmol m-2 d-1]",
]
fulweiler_sediment_obs = read_observation_csv(
    OBS_FILES["fulweiler_sediment"],
    usecols=sediment_columns,
    date_column="Date",
    numeric_columns=sediment_columns[2:],
)

In [ ]:
observational_data = {
    "NBFSMN": nbfsmn_obs,
    "PLT nitrogen": plt_nutrients_obs,
    "CHRP nutrients": chrp_obs,
    "PLT silicate": plt_si_obs,
    "PLT Secchi": plt_secchi_obs,
    "Oviatt production": oviatt_pp_obs,
    "Fulweiler sediment": fulweiler_sediment_obs,
}

observation_summary = pd.DataFrame.from_dict(
    {
        name: {
            "rows": len(frame),
            "columns": len(frame.columns),
            "memory_MB": frame.memory_usage(deep=True).sum() / 1_000_000,
        }
        for name, frame in observational_data.items()
    },
    orient="index",
).round({"memory_MB": 3})

print("[5/8] Observations loaded and standardized")
print(f"  Sources loaded: {len(observational_data)}")

display(observation_summary)

<a id="process-station-file"></a>
# 6. Process model output and match observations

This stage converts model time from UTC to local Eastern time, selects stations by name, calculates NBS-scale pH with TEOS-10 and PyCO2SYS, and matches model values using explicit site, depth, date, and month keys.

<div class="alert alert-info">
<strong>Traceability:</strong> Intermediate tables and validation summaries are intentionally retained so each aggregation and match can be inspected before costs are calculated.
</div>

## 6.1. Configuration and input validation

Station groups are configured in one place. To add an NBFSMN station later, add its abbreviation to `NBFSMN_STATIONS`; the observation filter and model selection will then use the same list.

In [ ]:
# Ordered station groups used by each observational dataset.
NBFSMN_STATIONS = ("BR", "MV", "QP", "TW", "GB", "GD", 
                   "PD", "CP", "NP", "PP", "MH", "SR")
SEDIMENT_STATIONS = ("GI", "GO", "GM", "CON")
CHRP_STATIONS = tuple(f"O{i}" for i in range(1, 17))

# The sediment CSV calls Conimicut "C"; model station names use "CON".
SEDIMENT_OBS_TO_MODEL = {
    "GI": "GI",
    "GO": "GO",
    "GM": "GM",
    "C": "CON",
}

# PLT samples in the original workflow were compared with the 08:00
# local model output. Keeping this setting explicit makes it easy to
# change if sampling-time information improves later.
PLT_SAMPLE_HOUR = 8
SECCHI_LIGHT_FRACTION = 0.15

REQUIRED_MODEL_VARIABLES = {
    "temp", "salt", "TIC", "alkalinity", "oxygen",
    "NO3", "NH4", "SiOH4", "kdPAR",
    "SOD", "benthic_flux_NO3", "benthic_flux_NH4",
    "h", "Cs_r", "lon_rho", "lat_rho", "ocean_time",
}

missing_model_variables = sorted(REQUIRED_MODEL_VARIABLES - set(sta.variables))
if missing_model_variables:
    raise KeyError(
        "Station file is missing required variables: "
        + ", ".join(missing_model_variables)
    )

n_vertical_levels = sta.sizes.get("s_rho")
if n_vertical_levels not in {10, 15}:
    raise ValueError(
        f"Expected 10 or 15 s_rho levels; found {n_vertical_levels}."
    )

required_station_names = (
    *NBFSMN_STATIONS,
    "PLT",
    *SEDIMENT_STATIONS,
    *CHRP_STATIONS,
)
missing_model_stations = [
    name for name in required_station_names
    if station_indices.get(name) is None
]

# Select every available station by its resolved index, then reindex to the
# complete requested list. Reindexing creates all-NaN model entries for a
# missing station while preserving the stable station coordinate expected by
# the cost-summary schema.
def select_named_stations(dataset, names):
    names = tuple(names)
    available_names = [
        name for name in names
        if station_indices.get(name) is not None
    ]
    available_indices = np.asarray(
        [station_indices[name] for name in available_names],
        dtype=int,
    )
    indexer = xr.DataArray(
        available_indices,
        dims="site",
        coords={"site": available_names},
    )
    selected = dataset.isel(station=indexer)
    return selected.reindex(site=list(names))

# Index 0 is always bottom; index -1 is surface for either 10 or 15 levels.
depth_indexer = xr.DataArray(
    [n_vertical_levels - 1, 0],
    dims="depth",
    coords={"depth": ["surface", "bottom"]},
    name="s_rho_index",
)

station_processing_config = pd.Series({
    "vertical_levels": n_vertical_levels,
    "surface_s_rho_index": n_vertical_levels - 1,
    "bottom_s_rho_index": 0,
    "NBFSMN_station_count": len(NBFSMN_STATIONS),
    "CHRP_station_count": len(CHRP_STATIONS),
    "sediment_station_count": len(SEDIMENT_STATIONS),
}, name="value")

station_processing_config

<a id="convert-model-time"></a>
## 6.2. Convert station time from UTC to US Eastern time

NetCDF cannot reliably preserve timezone-aware datetime coordinates. `ocean_time` is therefore stored as timezone-naive Eastern time after conversion, while the original timezone-naive UTC values remain available in `ocean_time_utc` for validation.

In [ ]:
def convert_utc_to_eastern(dataset, time_name="ocean_time"):
    '''Return a shallow copy with UTC model timestamps converted to Eastern time.'''
    original = pd.DatetimeIndex(pd.to_datetime(dataset[time_name].values))

    if original.tz is None:
        utc = original.tz_localize("UTC")
    else:
        utc = original.tz_convert("UTC")

    eastern = utc.tz_convert("America/New_York")

    # Store naive numpy datetimes while documenting their time zones.
    result = dataset.assign_coords(
        {
            time_name: eastern.tz_localize(None).to_numpy(),
            f"{time_name}_utc": (
                time_name,
                utc.tz_localize(None).to_numpy(),
            ),
        }
    )
    result[time_name].attrs.update({
        "timezone": "America/New_York",
        "source_timezone": "UTC",
    })
    result[f"{time_name}_utc"].attrs["timezone"] = "UTC"
    return result

sta_eastern = convert_utc_to_eastern(sta)

utc_index = pd.DatetimeIndex(sta_eastern["ocean_time_utc"].values)
eastern_index = pd.DatetimeIndex(sta_eastern["ocean_time"].values)
available_local_years = sorted(pd.Index(eastern_index.year).unique())
if runyear not in available_local_years:
    raise ValueError(
        f"Station file does not contain local-time records for {runyear}. "
        f"Available years: {available_local_years}. Check station_path."
    )
station_time_validation = pd.Series({
    "UTC start": utc_index.min(),
    "UTC end": utc_index.max(),
    "Eastern start": eastern_index.min(),
    "Eastern end": eastern_index.max(),
    "UTC timestamps": len(utc_index),
    # Repeated local times are expected during the autumn DST transition.
    "repeated Eastern timestamps": int(eastern_index.duplicated().sum()),
}, name="value")

station_time_validation

<a id="calculate-model-ph"></a>
## 6.3. Calculate NBS-scale pH with TEOS-10 and PyCO2SYS

The carbonate calculation is limited to NBFSMN stations and surface/bottom layers. Practical salinity and potential temperature are converted to TEOS-10 quantities before density is calculated. TIC and alkalinity are then converted from mmol m⁻³ to µmol kg⁻¹.

Depth is approximated as `Cs_r * h`; PyCO2SYS optional inputs retain their defaults.

In [ ]:
def calculate_nbs_ph(dataset):
    '''
    Add TEOS-10 seawater properties and PyCO2SYS NBS-scale pH.

    Expected dimensions are ocean_time, site, and depth. The returned
    dataset is loaded into memory so the calculation is performed once
    and all intermediate quantities remain available for validation.
    '''
    model = dataset[
        ["temp", "salt", "TIC", "alkalinity", "oxygen", "h", "Cs_r"]
    ].load()

    # Cs_r is a fractional vertical coordinate; h is positive bathymetry.
    z = model["Cs_r"] * model["h"]

    pressure = xr.apply_ufunc(
        gsw.p_from_z,
        z,
        dataset["lat_rho"],
    )
    pressure = xr.broadcast(pressure, model["temp"])[0].transpose(
        *model["temp"].dims
    )

    longitude = xr.broadcast(dataset["lon_rho"], model["temp"])[0].transpose(
        *model["temp"].dims
    )
    latitude = xr.broadcast(dataset["lat_rho"], model["temp"])[0].transpose(
        *model["temp"].dims
    )

    absolute_salinity = xr.apply_ufunc(
        gsw.SA_from_SP,
        model["salt"],
        pressure,
        longitude,
        latitude,
    )
    conservative_temperature = xr.apply_ufunc(
        gsw.CT_from_pt,
        absolute_salinity,
        model["temp"],
    )
    density = xr.apply_ufunc(
        gsw.rho,
        absolute_salinity,
        conservative_temperature,
        pressure,
    )
    insitu_temperature = xr.apply_ufunc(
        gsw.t_from_CT,
        absolute_salinity,
        conservative_temperature,
        pressure,
    )

    # mmol m-3 * 1000 umol mmol-1 / density kg m-3 = umol kg-1.
    tic_umol_kg = model["TIC"] * 1000 / density
    alkalinity_umol_kg = model["alkalinity"] * 1000 / density

    valid = (
        np.isfinite(tic_umol_kg)
        & np.isfinite(alkalinity_umol_kg)
        & np.isfinite(insitu_temperature)
        & np.isfinite(absolute_salinity)
        & np.isfinite(pressure)
        & (tic_umol_kg > 0)
        & (alkalinity_umol_kg > 0)
    )

    co2 = pyco2.sys(
        par1=alkalinity_umol_kg.where(valid).values,
        par1_type=1,
        par2=tic_umol_kg.where(valid).values,
        par2_type=2,
        temperature=insitu_temperature.where(valid).values,
        salinity=model["salt"].where(valid).values,
        pressure=pressure.where(valid).values,
    )

    model = model.assign(
        pressure_dbar=pressure,
        absolute_salinity=absolute_salinity,
        conservative_temperature=conservative_temperature,
        insitu_temperature=insitu_temperature,
        density=density,
        TIC_umol_kg=tic_umol_kg,
        alkalinity_umol_kg=alkalinity_umol_kg,
        pH_nbs=xr.DataArray(
            co2["pH_nbs"],
            coords=model["temp"].coords,
            dims=model["temp"].dims,
        ),
    )

    model["pressure_dbar"].attrs.update(units="dbar")
    model["absolute_salinity"].attrs.update(units="g kg-1")
    model["conservative_temperature"].attrs.update(units="degC")
    model["insitu_temperature"].attrs.update(units="degC")
    model["density"].attrs.update(units="kg m-3")
    model["TIC_umol_kg"].attrs.update(units="umol kg-1")
    model["alkalinity_umol_kg"].attrs.update(units="umol kg-1")
    model["pH_nbs"].attrs.update(scale="NBS", calculated_with="PyCO2SYS")
    return model

nbfsmn_model_layers = select_named_stations(
    sta_eastern,
    NBFSMN_STATIONS,
).isel(s_rho=depth_indexer)

# Practical salinity cannot be negative. Small negative values can occur as a
# numerical undershoot in very fresh model cells, but passing them to TEOS-10
# produces invalid Absolute and Conservative Temperature values. Treat these
# records as missing rather than clipping them to zero and inventing data.
negative_salinity_mask = (
    np.isfinite(nbfsmn_model_layers["salt"])
    & (nbfsmn_model_layers["salt"] < 0)
)
negative_salinity_count = int(negative_salinity_mask.sum().item())

negative_salinity_validation = (
    negative_salinity_mask
    .sum("ocean_time")
    .rename("masked_value_count")
    .to_dataframe()
    .reset_index()
)
negative_salinity_validation = negative_salinity_validation.loc[
    negative_salinity_validation["masked_value_count"] > 0
].reset_index(drop=True)
negative_salinity_validation["masked_value_count"] = (
    negative_salinity_validation["masked_value_count"].astype(int)
)

negative_salinity_counts_by_site_depth = {
    f"{row.site}:{row.depth}": int(row.masked_value_count)
    for row in negative_salinity_validation.itertuples(index=False)
}

if negative_salinity_count:
    print(
        "WARNING: Masked "
        f"{negative_salinity_count} negative practical-salinity values "
        "before model processing."
    )
    for row in negative_salinity_validation.itertuples(index=False):
        print(
            f"  {row.site} {str(row.depth).title()}: "
            f"{row.masked_value_count}"
        )
else:
    print("Salinity quality control: no negative values found.")

# This sanitized array feeds both the salinity comparison and all TEOS-10 /
# PyCO2SYS calculations, so invalid values cannot influence daily averages.
nbfsmn_model_layers = nbfsmn_model_layers.assign(
    salt=nbfsmn_model_layers["salt"].where(~negative_salinity_mask)
)
nbfsmn_model_layers["salt"].attrs.update({
    "quality_control": "finite values below 0 PSU replaced with NaN",
    "negative_values_masked": negative_salinity_count,
})

nbfsmn_model_carbonate = calculate_nbs_ph(nbfsmn_model_layers)

if bool((nbfsmn_model_carbonate["salt"] < 0).any().item()):
    raise AssertionError("Negative salinity remained after quality control.")

carbonate_validation = pd.DataFrame({
    "minimum": {
        name: float(nbfsmn_model_carbonate[name].min(skipna=True))
        for name in [
            "pressure_dbar", "density", "TIC_umol_kg",
            "alkalinity_umol_kg", "pH_nbs",
        ]
    },
    "maximum": {
        name: float(nbfsmn_model_carbonate[name].max(skipna=True))
        for name in [
            "pressure_dbar", "density", "TIC_umol_kg",
            "alkalinity_umol_kg", "pH_nbs",
        ]
    },
})

carbonate_validation

## 6.4. Matching helpers

Observations remain the left-hand side of every merge. Consequently, unmatched observations are retained and visible instead of being silently discarded.

In [ ]:
def add_local_date(dataset, coordinate_name="sample_date"):
    '''Attach normalized local calendar dates to an xarray dataset.'''
    dates = pd.DatetimeIndex(dataset["ocean_time"].values).normalize()
    return dataset.assign_coords(
        {coordinate_name: ("ocean_time", dates.to_numpy(dtype="datetime64[ns]"))}
    )

def daily_model_frame(dataset, variables):
    '''Return daily model means as a tidy DataFrame.'''
    with_dates = add_local_date(dataset)
    daily = with_dates[list(variables)].groupby("sample_date").mean("ocean_time")
    return daily[list(variables)].to_dataframe().reset_index()

def model_at_local_hour(dataset, variables, hour=PLT_SAMPLE_HOUR):
    '''Return model records at one local clock hour as a tidy DataFrame.'''
    local_time = pd.DatetimeIndex(dataset["ocean_time"].values)
    selected = dataset[list(variables)].isel(ocean_time=(local_time.hour == hour))
    frame = selected[list(variables)].to_dataframe().reset_index()
    return frame.rename(columns={"ocean_time": "sample_time"})

def add_match_coverage(rows, source, frame, pairs):
    '''Append per-variable observation/model match counts to rows.'''
    for target, observation_column, model_column in pairs:
        has_observation = frame[observation_column].notna()
        matched = has_observation & frame[model_column].notna()
        n_observations = int(has_observation.sum())
        n_matched = int(matched.sum())
        rows.append({
            "source": source,
            "target": target,
            "observations": n_observations,
            "matched": n_matched,
            "unmatched": n_observations - n_matched,
            "matched_percent": (
                100 * n_matched / n_observations
                if n_observations else np.nan
            ),
        })

## 6.5. NBFSMN daily observations and model results

Both observations and model output are averaged by local calendar date, site, and depth. No fixed 365-day arrays are used, so missing days and leap years remain explicit.

In [ ]:
nbfsmn_obs_for_matching = nbfsmn_obs.loc[
    nbfsmn_obs["site"].isin(NBFSMN_STATIONS)
].copy()
nbfsmn_obs_for_matching["sample_date"] = (
    nbfsmn_obs_for_matching["date"].dt.normalize()
)

nbfsmn_obs_daily = (
    nbfsmn_obs_for_matching
    .groupby(["sample_date", "site", "depth"], as_index=False)
    .agg(
        temperature_obs=("T", "mean"),
        temperature_obs_std=("T", "std"),
        salinity_obs=("S", "mean"),
        salinity_obs_std=("S", "std"),
        oxygen_obs=("DO [mmol/m3]", "mean"),
        oxygen_obs_std=("DO [mmol/m3]", "std"),
        pH_obs=("pH", "mean"),
        pH_obs_std=("pH", "std"),
    )
)

nbfsmn_model_daily = daily_model_frame(
    nbfsmn_model_carbonate,
    ["temp", "salt", "oxygen", "pH_nbs"],
).rename(columns={
    "temp": "temperature_model",
    "salt": "salinity_model",
    "oxygen": "oxygen_model",
    "pH_nbs": "pH_model",
})

nbfsmn_matched = nbfsmn_obs_daily.merge(
    nbfsmn_model_daily,
    on=["sample_date", "site", "depth"],
    how="left",
    validate="one_to_one",
    indicator="model_match",
)

nbfsmn_matched.head()

## 6.6. PLT and CHRP nutrient observations

PLT nutrient and silicate observations are compared with the 08:00 local model record on each sampling date. CHRP observations are mapped explicitly from CSV station numbers to `O1`–`O16` and compared with daily model means.

In [ ]:
# PLT model values at the configured local sampling hour.
plt_model_layers = select_named_stations(sta_eastern, ["PLT"]).isel(
    s_rho=depth_indexer
)
plt_model_at_sample_time = model_at_local_hour(
    plt_model_layers,
    ["NO3", "NH4", "SiOH4"],
).drop(columns="site")

# Average duplicate measurements on the same date and depth, then attach
# the documented local sampling hour used by the original workflow.
plt_nutrients_daily = (
    plt_nutrients_obs
    .assign(sample_date=plt_nutrients_obs["Date"].dt.normalize())
    .groupby(["sample_date", "depth"], as_index=False)
    .agg(
        NO3_obs=("NO3 (uM)", "mean"),
        NH4_obs=("NH4 (uM)", "mean"),
    )
)
plt_nutrients_daily["sample_time"] = (
    plt_nutrients_daily["sample_date"]
    + pd.Timedelta(hours=PLT_SAMPLE_HOUR)
)

plt_nutrients_matched = plt_nutrients_daily.merge(
    plt_model_at_sample_time.rename(columns={
        "NO3": "NO3_model",
        "NH4": "NH4_model",
    })[["sample_time", "depth", "NO3_model", "NH4_model"]],
    on=["sample_time", "depth"],
    how="left",
    validate="one_to_one",
    indicator="model_match",
)

# Convert wide surface/bottom Si observations to the same tidy depth key.
plt_si_long = (
    plt_si_obs
    .melt(
        id_vars="Date",
        value_vars=["Surface Si", "Bottom Si"],
        var_name="depth",
        value_name="Si_obs",
    )
    .replace({"depth": {"Surface Si": "surface", "Bottom Si": "bottom"}})
)
plt_si_daily = (
    plt_si_long
    .assign(sample_date=plt_si_long["Date"].dt.normalize())
    .groupby(["sample_date", "depth"], as_index=False)
    .agg(Si_obs=("Si_obs", "mean"))
)
plt_si_daily["sample_time"] = (
    plt_si_daily["sample_date"]
    + pd.Timedelta(hours=PLT_SAMPLE_HOUR)
)

plt_si_matched = plt_si_daily.merge(
    plt_model_at_sample_time.rename(columns={"SiOH4": "Si_model"})[
        ["sample_time", "depth", "Si_model"]
    ],
    on=["sample_time", "depth"],
    how="left",
    validate="one_to_one",
    indicator="model_match",
)

# CHRP Station=1 maps to O1, ..., Station=16 maps to O16.
chrp_daily = chrp_obs.copy()
chrp_daily["site"] = (
    pd.to_numeric(chrp_daily["Station"], errors="coerce")
    .astype("Int64")
    .map({i: f"O{i}" for i in range(1, 17)})
)
chrp_daily["sample_date"] = chrp_daily["Date"].dt.normalize()
chrp_daily["depth"] = "surface"
chrp_daily = (
    chrp_daily
    .groupby(["sample_date", "site", "depth"], as_index=False)
    .agg(
        CHRP_NO3_obs=("NO3 [mmol m-3]", "mean"),
        CHRP_NH4_obs=("NH3 [mmol m-3]", "mean"),
        CHRP_Si_obs=("Si [mmol m-3]", "mean"),
    )
)

chrp_model_layers = select_named_stations(
    sta_eastern,
    CHRP_STATIONS,
).isel(s_rho=depth_indexer)
chrp_model_daily = daily_model_frame(
    chrp_model_layers,
    ["NO3", "NH4", "SiOH4"],
).rename(columns={
    "NO3": "CHRP_NO3_model",
    "NH4": "CHRP_NH4_model",
    "SiOH4": "CHRP_Si_model",
})

chrp_matched = chrp_daily.merge(
    chrp_model_daily,
    on=["sample_date", "site", "depth"],
    how="left",
    validate="one_to_one",
    indicator="model_match",
)

plt_nutrients_matched.head()

## 6.7. Secchi depth and sediment fluxes

Secchi depth is calculated at PLT from the Beer–Lambert relationship using the configured 15% light threshold. Sediment observations and model output are averaged by site and month.

In [ ]:
# Secchi depth: I(z)/I(0) = exp(-k z), with k positive downward.
plt_optics = select_named_stations(sta_eastern, ["PLT"]).squeeze("site")
attenuation = plt_optics["kdPAR"].max("s_rho")
secchi_model = (
    -np.log(SECCHI_LIGHT_FRACTION) / attenuation
).where(attenuation > 0)
secchi_model.name = "Secchi_model"
secchi_model_at_sample_time = model_at_local_hour(
    secchi_model.to_dataset(),
    ["Secchi_model"],
)[["sample_time", "Secchi_model"]]

plt_secchi_daily = (
    plt_secchi_obs
    .assign(sample_date=plt_secchi_obs["Date"].dt.normalize())
    .groupby("sample_date", as_index=False)
    .agg(Secchi_obs=("Secchi Depth", "mean"))
)
plt_secchi_daily["sample_time"] = (
    plt_secchi_daily["sample_date"]
    + pd.Timedelta(hours=PLT_SAMPLE_HOUR)
)
plt_secchi_matched = plt_secchi_daily.merge(
    secchi_model_at_sample_time,
    on="sample_time",
    how="left",
    validate="one_to_one",
    indicator="model_match",
)

# Monthly sediment observations. Standard deviations remain available
# for later weighting and diagnostics.
sediment_obs_monthly = fulweiler_sediment_obs.copy()
sediment_obs_monthly["site"] = sediment_obs_monthly["Site"].map(
    SEDIMENT_OBS_TO_MODEL
)
if sediment_obs_monthly["site"].isna().any():
    unknown = sorted(
        sediment_obs_monthly.loc[
            sediment_obs_monthly["site"].isna(), "Site"
        ].dropna().unique()
    )
    raise KeyError(f"Unmapped sediment observation sites: {unknown}")
sediment_obs_monthly["month"] = sediment_obs_monthly["Date"].dt.month
sediment_obs_monthly = (
    sediment_obs_monthly
    .groupby(["site", "month"], as_index=False)
    .agg(
        SOD_obs=("O2 [mmol m-2 d-1]", "mean"),
        SOD_obs_std=("O2 [mmol m-2 d-1]", "std"),
        benthic_NO3_obs=("NO3 [mmol m-2 d-1]", "mean"),
        benthic_NO3_obs_std=("NO3 [mmol m-2 d-1]", "std"),
        benthic_NH4_obs=("NH4 [mmol m-2 d-1]", "mean"),
        benthic_NH4_obs_std=("NH4 [mmol m-2 d-1]", "std"),
    )
)

sediment_model = select_named_stations(
    sta_eastern,
    SEDIMENT_STATIONS,
)[["SOD", "benthic_flux_NO3", "benthic_flux_NH4"]]
sediment_model_monthly = (
    sediment_model
    .groupby("ocean_time.month")
    .mean("ocean_time")
    .to_dataframe()
    .reset_index()
    .rename(columns={
        "SOD": "SOD_model",
        "benthic_flux_NO3": "benthic_NO3_model",
        "benthic_flux_NH4": "benthic_NH4_model",
    })
)

sediment_matched = sediment_obs_monthly.merge(
    sediment_model_monthly,
    on=["site", "month"],
    how="left",
    validate="one_to_one",
    indicator="model_match",
)

sediment_matched.head()

## 6.8. Oviatt annual primary production

Annual net daytime production is read from the production file associated with the model run. Model and observational values are matched explicitly by Oviatt station number rather than row order.

In [ ]:
OVIATT_OBS_COLUMN = "C-14 Production Estimate (gC m-2 y-1)"
OVIATT_MODEL_COLUMN = "Annual Net Daytime Production (gC/m^2/yr)"

if production_file_missing:
    # The production diagnostic is optional. Preserve all 16 legacy stations
    # and make the absent model result explicit rather than failing the run.
    oviatt_pp_model = pd.DataFrame({
        "Station Number": np.arange(1, 17),
        OVIATT_MODEL_COLUMN: np.full(16, np.nan),
    })
else:
    # If the file exists but has an incompatible schema, read_csv should raise:
    # that indicates a malformed result rather than an intentionally absent one.
    oviatt_pp_model = pd.read_csv(
        PPPATH,
        usecols=["Station Number", OVIATT_MODEL_COLUMN],
    )
    oviatt_pp_model[["Station Number", OVIATT_MODEL_COLUMN]] = (
        oviatt_pp_model[["Station Number", OVIATT_MODEL_COLUMN]]
        .apply(pd.to_numeric, errors="coerce")
    )

oviatt_pp_observed = oviatt_pp_obs.rename(columns={
    OVIATT_OBS_COLUMN: "primary_production_obs",
})
oviatt_pp_modeled = oviatt_pp_model.rename(columns={
    OVIATT_MODEL_COLUMN: "primary_production_model",
})

for label, frame in {
    "observations": oviatt_pp_observed,
    "model production": oviatt_pp_modeled,
}.items():
    if frame["Station Number"].isna().any():
        raise ValueError(f"{label} contain a missing station number.")
    if frame["Station Number"].duplicated().any():
        duplicates = frame.loc[
            frame["Station Number"].duplicated(keep=False),
            "Station Number",
        ].tolist()
        raise ValueError(f"{label} contain duplicate stations: {duplicates}")

oviatt_pp_matched = oviatt_pp_observed.merge(
    oviatt_pp_modeled,
    on="Station Number",
    how="left",
    validate="one_to_one",
    indicator="model_match",
)
oviatt_pp_matched["site"] = (
    "O" + oviatt_pp_matched["Station Number"].astype("Int64").astype(str)
)

oviatt_pp_matched

<a id="match-coverage"></a>
## 6.9. Match coverage and validation

Review this table before interpreting costs. Low coverage indicates a station, date, time, or depth alignment problem rather than silently reducing the sample size.

In [ ]:
coverage_rows = []

add_match_coverage(coverage_rows, "NBFSMN", nbfsmn_matched, [
    ("temperature", "temperature_obs", "temperature_model"),
    ("salinity", "salinity_obs", "salinity_model"),
    ("oxygen", "oxygen_obs", "oxygen_model"),
    ("pH (NBS)", "pH_obs", "pH_model"),
])
add_match_coverage(coverage_rows, "PLT", plt_nutrients_matched, [
    ("NO3", "NO3_obs", "NO3_model"),
    ("NH4", "NH4_obs", "NH4_model"),
])
add_match_coverage(coverage_rows, "PLT", plt_si_matched, [
    ("Si", "Si_obs", "Si_model"),
])
add_match_coverage(coverage_rows, "CHRP", chrp_matched, [
    ("NO3", "CHRP_NO3_obs", "CHRP_NO3_model"),
    ("NH4", "CHRP_NH4_obs", "CHRP_NH4_model"),
    ("Si", "CHRP_Si_obs", "CHRP_Si_model"),
])
add_match_coverage(coverage_rows, "PLT", plt_secchi_matched, [
    ("Secchi depth", "Secchi_obs", "Secchi_model"),
])
add_match_coverage(coverage_rows, "Fulweiler", sediment_matched, [
    ("SOD", "SOD_obs", "SOD_model"),
    ("benthic NO3", "benthic_NO3_obs", "benthic_NO3_model"),
    ("benthic NH4", "benthic_NH4_obs", "benthic_NH4_model"),
])
add_match_coverage(coverage_rows, "Oviatt", oviatt_pp_matched, [
    ("primary production", "primary_production_obs", "primary_production_model"),
])

match_coverage = (
    pd.DataFrame(coverage_rows)
    .set_index(["source", "target"])
    .sort_index()
)
match_coverage["matched_percent"] = match_coverage["matched_percent"].round(1)

matched_observation_tables = {
    "NBFSMN": nbfsmn_matched,
    "PLT nitrogen": plt_nutrients_matched,
    "PLT silicate": plt_si_matched,
    "CHRP nutrients": chrp_matched,
    "PLT Secchi": plt_secchi_matched,
    "Fulweiler sediment": sediment_matched,
    "Oviatt production": oviatt_pp_matched,
}

print("[6/8] Model-observation matching complete")
print(f"  Targets audited: {len(match_coverage)}")

display(match_coverage)

<a id="construct-cost-summary"></a>
# 7. Construct the legacy-compatible cost-summary dataset

The cost-summary NetCDF is a public interface used by other analysis scripts. Variable names, coordinate names, and dimension order therefore remain compatible with `LHS1_2005.nc` and `C5_2006.nc`.

Internally, labeled tables are reindexed onto explicit coordinates, missing entries remain `NaN`, and compatibility assertions run before saving.

<a id="metadata-definitions"></a>
## 7.1. Metadata definitions

Metadata record scientific provenance and the exact treatment applied in this notebook. Legacy concepts are retained while ambiguous descriptions are clarified.

In [ ]:
NBFSMN_LINK = (
    "https://web.uri.edu/wp-content/uploads/sites/916/"
    "nbfsmn_daily_means_thru_2019.csv"
)
PLT_LINK = "https://web.uri.edu/gso/research/plankton/data/"
CHRP_LINK = (
    "https://web.uri.edu/gso/research/"
    "marine-ecosystems-research-laboratory/datasets/"
)
FULWEILER_LINK = (
    "https://docs.google.com/spreadsheets/d/"
    "1y12AWpDzeZxax9kRjpIZq5XPI5fUJvStbCqbYxu9y8Y/"
    "edit?gid=761968705#gid=761968705"
)
PARAMETER_LINK = (
    "https://docs.google.com/spreadsheets/d/"
    "1iDZJ_ArikoeY8c2dsS_Ef-qasBZLU_o0S6_lvbprZgE/"
    "edit?gid=0#gid=0"
)

OVIATT_CITATION = (
    "Oviatt, C., Keller, A., & Reed, L. (2002). Annual primary "
    "production in Narragansett Bay with no bay-wide winter-spring "
    "phytoplankton bloom. Estuarine, Coastal and Shelf Science, "
    "54(6), 1013-1026."
)
CHRP_CITATION = (
    "Reed, Laura and Candace Oviatt. 2006-2019. Marine Ecosystem "
    "Research Laboratory, Graduate School of Oceanography, URI, "
    "Narragansett, R.I."
)
FULWEILER_CITATION = (
    "Fulweiler, R. W. (2007). The impact of climate change on "
    "benthic-pelagic coupling and the biogeochemical cycling of "
    "Narragansett Bay, R.I. (PhD). University of Rhode Island."
)

# Region names are shared by the Oviatt production and CHRP stations.
OVIATT_REGIONS = {
    1: "West Passage", 2: "West Passage", 3: "RI Sound",
    4: "East Passage", 5: "East Passage", 6: "East Passage",
    7: "Mt Hope Bay", 8: "East Passage",
    9: "Providence River", 10: "Providence River",
    11: "Providence River", 12: "Providence River",
    13: "Providence River", 14: "West Passage",
    15: "Greenwich Bay", 16: "West Passage",
}

# Derive the observational sediment date range from the data rather than
# retaining a hard-coded year range that can become stale.
sediment_year_values = sorted(
    fulweiler_sediment_obs["Date"].dropna().dt.year.unique()
)
sediment_year_range = (
    f"{sediment_year_values[0]}-{sediment_year_values[-1]}"
    if sediment_year_values else "unknown"
)

def common_observation_attrs(units, long_name, source, treatment):
    '''Build consistent provenance metadata for an observed variable.'''
    return {
        "units": units,
        "long_name": long_name,
        "source": source,
        "data_treatment": treatment,
    }

def common_model_attrs(units, long_name, treatment):
    '''Build consistent processing metadata for a modeled variable.'''
    return {
        "units": units,
        "long_name": long_name,
        "source": str(station_path),
        "source_timezone": "UTC",
        "comparison_timezone": "America/New_York",
        "data_treatment": treatment,
    }

VARIABLE_ATTRS = {
    "pH_obs": common_observation_attrs(
        "NBS scale", "observed pH", NBFSMN_LINK,
        f"Daily averages for {runyear}."
    ),
    "pH_mod": {
        **common_model_attrs(
            "NBS scale", "modeled pH",
            "Daily local-time averages at observation sites."
        ),
        "calculation": (
            "PyCO2SYS from total alkalinity and TIC after TEOS-10 "
            "salinity, temperature, pressure, and density conversion."
        ),
        "carbonate_pair": "total alkalinity (type 1), DIC (type 2)",
    },
    "oxygen_obs": common_observation_attrs(
        "mmol m-3", "observed dissolved oxygen", NBFSMN_LINK,
        f"Daily averages for {runyear}; converted from mg L-1."
    ),
    "oxygen_mod": common_model_attrs(
        "mmol m-3", "modeled dissolved oxygen",
        "Daily local-time averages at observation sites."
    ),
    "temp_obs": common_observation_attrs(
        "degree_Celsius", "observed water temperature", NBFSMN_LINK,
        f"Daily averages for {runyear}."
    ),
    "temp_mod": common_model_attrs(
        "degree_Celsius", "modeled potential temperature",
        "Daily local-time averages at observation sites."
    ),
    "salt_obs": common_observation_attrs(
        "PSU", "observed practical salinity", NBFSMN_LINK,
        f"Daily averages for {runyear}."
    ),
    "salt_mod": {
        **common_model_attrs(
            "PSU", "modeled practical salinity",
            "Daily local-time averages at observation sites."
        ),
        "quality_control": (
            "Finite practical-salinity values below 0 PSU were replaced "
            "with NaN before TEOS-10 conversion and daily averaging."
        ),
        "negative_values_masked": negative_salinity_count,
        "masked_counts_by_site_depth": json.dumps(
            negative_salinity_counts_by_site_depth
        ),
    },
}

for nutrient, name in {
    "NO3": "nitrate", "NH4": "ammonium", "Si": "silicate",
}.items():
    VARIABLE_ATTRS[f"{nutrient}_obs"] = common_observation_attrs(
        "mmol m-3", f"observed {name}", PLT_LINK,
        "Measurements averaged by sampling date and depth."
    )
    VARIABLE_ATTRS[f"{nutrient}_mod"] = common_model_attrs(
        "mmol m-3", f"modeled {name}",
        f"Model value at {PLT_SAMPLE_HOUR:02d}:00 local time on each sampling date."
    )

VARIABLE_ATTRS.update({
    "SD_obs": common_observation_attrs(
        "m", "observed Secchi depth", PLT_LINK,
        "Measurements averaged by sampling date."
    ),
    "SD_mod": common_model_attrs(
        "m", "modeled Secchi depth",
        f"Value at {PLT_SAMPLE_HOUR:02d}:00 local time; Beer-Lambert "
        f"depth at {SECCHI_LIGHT_FRACTION:.0%} of surface light using "
        "the maximum vertical kdPAR."
    ),
})

for nutrient, name in {
    "NO3": "nitrate", "NH4": "ammonium", "Si": "silicate",
}.items():
    VARIABLE_ATTRS[f"{nutrient}_chrp_obs"] = {
        **common_observation_attrs(
            "mmol m-3", f"observed CHRP {name}", CHRP_LINK,
            "Daily station means; observations are surface-only and "
            "the legacy bottom layer is filled with NaN."
        ),
        "citation": CHRP_CITATION,
    }
    VARIABLE_ATTRS[f"{nutrient}_chrp_mod"] = common_model_attrs(
        "mmol m-3", f"modeled CHRP {name}",
        "Daily local-time averages at Oviatt stations."
    )

VARIABLE_ATTRS.update({
    "PP_obs": {
        **common_observation_attrs(
            "gC m-2 yr-1", "observed annual net daytime primary production",
            str(OBS_FILES["oviatt_pp"]),
            "Depth-integrated C-14 incubation estimates from 1997-1998."
        ),
        "citation": OVIATT_CITATION,
    },
    "PP_mod": {
        "units": "gC m-2 yr-1",
        "long_name": "modeled annual net daytime primary production",
        "source": str(PPPATH),
        "data_treatment": (
            "Depth-integrated net daytime production of both model "
            "phytoplankton classes at Oviatt stations."
        ),
    },
})

for legacy_name, chemical, direction in [
    ("sed_o2", "oxygen", "positive into sediment"),
    ("sed_no3", "nitrate", "positive out of sediment"),
    ("sed_nh4", "ammonium", "positive out of sediment"),
]:
    VARIABLE_ATTRS[f"{legacy_name}_obs"] = {
        **common_observation_attrs(
            "mmol m-2 d-1", f"observed sediment {chemical} flux",
            FULWEILER_LINK,
            "Monthly site means across available core incubations; "
            f"source years {sediment_year_range}."
        ),
        "citation": FULWEILER_CITATION,
        "sign_convention": "source sign convention retained",
    }
    VARIABLE_ATTRS[f"{legacy_name}_mod"] = {
        **common_model_attrs(
            "mmol m-2 d-1", f"modeled sediment {chemical} flux",
            "Monthly local-time average for each site."
        ),
        "sign_convention": direction,
    }

COST_ATTRS = {
    "units": "nondimensional",
    "calculation_status": "pending",
    "missing_value_meaning": "cost has not yet been calculated",
}

## 7.2. Parameter values and structured parameter metadata

`param_vals(Parameters)` is preserved for downstream compatibility. Descriptions and units are also stored as aligned string coordinates so newer code does not need to parse stringified arrays from attributes.

In [ ]:
PARAMETER_LIST_PATH = SPOS_DIR / "ParameterList.csv"
if not PARAMETER_LIST_PATH.is_file():
    raise FileNotFoundError(
        f"Parameter metadata file does not exist: {PARAMETER_LIST_PATH}"
    )

parameter_table = pd.read_csv(PARAMETER_LIST_PATH).iloc[:74].copy()
parameter_table.columns = parameter_table.columns.str.strip()
parameter_table["Parameter Name"] = (
    parameter_table["Parameter Name"].astype(str).str.strip()
)

if parameter_table["Parameter Name"].duplicated().any():
    duplicates = parameter_table.loc[
        parameter_table["Parameter Name"].duplicated(keep=False),
        "Parameter Name",
    ].tolist()
    raise ValueError(f"Duplicate parameter names: {duplicates}")

missing_parameter_variables = [
    name for name in parameter_table["Parameter Name"]
    if name not in sta.variables
]
if missing_parameter_variables:
    raise KeyError(
        "Station file is missing parameter variables: "
        + ", ".join(missing_parameter_variables)
    )

def first_parameter_value(data_array):
    '''Return the legacy scalar representation of a model parameter.'''
    values = np.asarray(data_array.values).reshape(-1)
    return float(values[0]) if values.size else np.nan

parameter_names = parameter_table["Parameter Name"].to_numpy(dtype=str)
parameter_descriptions = (
    parameter_table["Description"].fillna("").to_numpy(dtype=str)
)
parameter_units = parameter_table["Units"].fillna("").to_numpy(dtype=str)
parameter_values = np.asarray([
    first_parameter_value(sta[name]) for name in parameter_names
], dtype=float)

if not (
    len(parameter_names)
    == len(parameter_descriptions)
    == len(parameter_units)
    == len(parameter_values)
):
    raise ValueError("Parameter names, metadata, and values are misaligned.")

parameter_note = (
    "For vector-valued partitioning coefficients and decay rates, "
    "param_vals retains the first value for legacy compatibility. "
    "Consult the parameter spreadsheet for complete vectors."
)
parameter_attrs = {
    "long_name": "model parameter values",
    "source": str(station_path),
    "value_selection": "first element retained for vector-valued parameters",
    "parameter_count": len(parameter_names),
    "parameter_spreadsheet": PARAMETER_LINK,
    # Preserve the legacy attributes used by existing readers.
    "Description": str(parameter_descriptions),
    "Units": str(parameter_units),
    "Notes": parameter_note,
    "Link to spreadsheet": PARAMETER_LINK,
}

parameter_validation = pd.DataFrame({
    "parameter": parameter_names,
    "value": parameter_values,
    "units": parameter_units,
    "description": parameter_descriptions,
})

parameter_validation.head()

## 7.3. Labeled array-construction helpers

Every helper below works from named columns and coordinates. No result depends on incidental DataFrame row order, and empty input tables produce correctly shaped all-`NaN` arrays.

In [ ]:
def table_to_dataarray(frame, value_column, dimensions, coordinates):
    '''Reindex one tidy table column onto an explicit xarray grid.'''
    dimensions = tuple(dimensions)
    template = xr.DataArray(
        np.full(
            tuple(len(coordinates[dim]) for dim in dimensions),
            np.nan,
            dtype=float,
        ),
        dims=dimensions,
        coords={dim: coordinates[dim] for dim in dimensions},
        name=value_column,
    )

    if frame.empty:
        return template

    required_columns = {*dimensions, value_column}
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise KeyError(
            f"Cannot build {value_column!r}; missing columns: {missing_columns}"
        )

    values = frame[list(dimensions) + [value_column]].copy()
    duplicate_keys = values.duplicated(list(dimensions), keep=False)
    if duplicate_keys.any():
        duplicate_rows = values.loc[duplicate_keys, list(dimensions)]
        raise ValueError(
            f"Duplicate coordinate keys while building {value_column!r}:\n"
            f"{duplicate_rows.head()}"
        )

    indexed = values.set_index(list(dimensions))[value_column].to_xarray()
    return (
        indexed
        .reindex({dim: coordinates[dim] for dim in dimensions})
        .transpose(*dimensions)
        .astype(float)
    )

def title_depth(frame):
    '''Return a copy using legacy Surface/Bottom coordinate labels.'''
    result = frame.copy()
    result["Depth"] = result["depth"].str.title()
    return result

def station_number_from_site(site_series):
    '''Convert O1...O16 labels to legacy integer Station coordinates.'''
    return pd.to_numeric(
        site_series.astype(str).str.removeprefix("O"),
        errors="coerce",
    ).astype("Int64")

## 7.4. Reindex observations and model results onto the legacy schema

The legacy schema uses separate time coordinates for NBFSMN days, PLT nitrogen, PLT silicate, CHRP, and Secchi depth. Keeping these coordinates separate avoids forcing unrelated datasets onto a shared sampling calendar.

In [ ]:
# ------------------------------
# Shared legacy coordinates
# ------------------------------
site_values = list(NBFSMN_STATIONS)
depth_values = ["Surface", "Bottom"]
day_values = np.arange(1, n_days + 1)
month_values = np.arange(1, 13)
station_values = np.arange(1, 17)
loc_values = np.arange(1, len(SEDIMENT_STATIONS) + 1)

# ------------------------------
# NBFSMN: sparse observations, complete daily model record
# ------------------------------
nbfsmn_obs_grid = title_depth(nbfsmn_obs_daily.rename(columns={"site": "Site"}))
nbfsmn_obs_grid["Day"] = nbfsmn_obs_grid["sample_date"].dt.dayofyear

nbfsmn_model_grid = title_depth(
    nbfsmn_model_daily.rename(columns={"site": "Site"})
)
nbfsmn_model_grid = nbfsmn_model_grid.loc[
    nbfsmn_model_grid["sample_date"].dt.year.eq(runyear)
].copy()
nbfsmn_model_grid["Day"] = nbfsmn_model_grid["sample_date"].dt.dayofyear

nbfsmn_coordinates = {
    "Site": site_values,
    "Depth": depth_values,
    "Day": day_values,
}

# ------------------------------
# PLT nitrogen and silicate sampling dates
# ------------------------------
nitrogen_grid = title_depth(plt_nutrients_matched)
nitrogen_grid["Date"] = pd.to_datetime(nitrogen_grid["sample_time"])
date_values = np.sort(nitrogen_grid["Date"].dropna().unique())

silicate_grid = title_depth(plt_si_matched)
silicate_grid["DateSi"] = pd.to_datetime(silicate_grid["sample_time"])
date_si_values = np.sort(silicate_grid["DateSi"].dropna().unique())

# ------------------------------
# Secchi sampling dates
# ------------------------------
# The legacy Week coordinate contains dates with an actual Secchi
# observation; a source row whose Secchi value is missing is excluded.
secchi_grid = plt_secchi_matched.loc[
    plt_secchi_matched["Secchi_obs"].notna()
].copy()
secchi_grid["Week"] = pd.to_datetime(secchi_grid["sample_time"])
week_values = np.sort(secchi_grid["Week"].dropna().unique())

# ------------------------------
# CHRP: surface-only observations, two-depth model results
# ------------------------------
chrp_date_values = np.sort(chrp_daily["sample_date"].dropna().unique())

chrp_obs_grid = title_depth(chrp_daily)
chrp_obs_grid["Station"] = station_number_from_site(chrp_obs_grid["site"])
chrp_obs_grid["DateCHRP"] = pd.to_datetime(chrp_obs_grid["sample_date"])

chrp_model_grid = title_depth(chrp_model_daily)
chrp_model_grid["Station"] = station_number_from_site(
    chrp_model_grid["site"]
)
chrp_model_grid["DateCHRP"] = pd.to_datetime(
    chrp_model_grid["sample_date"]
)

# ------------------------------
# Annual production and monthly sediment fluxes
# ------------------------------
production_grid = oviatt_pp_matched.rename(
    columns={"Station Number": "Station"}
).copy()
production_grid["Station"] = production_grid["Station"].astype("Int64")

sediment_site_to_loc = {
    site: index + 1 for index, site in enumerate(SEDIMENT_STATIONS)
}
sediment_obs_grid = sediment_obs_monthly.copy()
sediment_obs_grid["Loc"] = sediment_obs_grid["site"].map(
    sediment_site_to_loc
)
sediment_obs_grid["Month"] = sediment_obs_grid["month"]

sediment_model_grid = sediment_model_monthly.copy()
sediment_model_grid["Loc"] = sediment_model_grid["site"].map(
    sediment_site_to_loc
)
sediment_model_grid["Month"] = sediment_model_grid["month"]

legacy_coordinate_sizes = pd.Series({
    "Site": len(site_values),
    "Depth": len(depth_values),
    "Day": len(day_values),
    "Date": len(date_values),
    "DateSi": len(date_si_values),
    "DateCHRP": len(chrp_date_values),
    "Week": len(week_values),
    "Station": len(station_values),
    "Loc": len(loc_values),
    "Month": len(month_values),
    "Parameters": len(parameter_names),
}, name="size")

legacy_coordinate_sizes

## 7.5. Assemble the cost-summary dataset

Cost variables are created now with `NaN` values and `calculation_status="pending"`. This makes an unfinished cost distinguishable from a valid zero cost while preserving every legacy variable expected by downstream scripts.

In [ ]:
site_attrs = {
    "Description": "Narragansett Bay Fixed Station Monitoring Sites",
    **{site: str(i) for i, site in enumerate(site_values)},
}
depth_attrs = {"Surface": "0", "Bottom": "1"}
oviatt_station_attrs = {
    f"Station {number}": region
    for number, region in OVIATT_REGIONS.items()
}
nutrient_station_attrs = {
    "PLT": "Narragansett Bay Long Term Phytoplankton Time Series",
    **oviatt_station_attrs,
}
fulweiler_location_attrs = {
    "Description": "sites of nutrient incubations in Fulweiler (2007)",
    "GI": "Greenwich Bay Inner",
    "GO": "Greenwich Bay Outer",
    "GM": "Greenwich Bay Mid",
    "CON": "Conimicut Point",
}

data_vars = {
    # NBFSMN observations and complete daily model records
    "pH_obs": table_to_dataarray(
        nbfsmn_obs_grid, "pH_obs", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),
    "pH_mod": table_to_dataarray(
        nbfsmn_model_grid, "pH_model", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),
    "oxygen_obs": table_to_dataarray(
        nbfsmn_obs_grid, "oxygen_obs", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),
    "oxygen_mod": table_to_dataarray(
        nbfsmn_model_grid, "oxygen_model", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),
    "temp_obs": table_to_dataarray(
        nbfsmn_obs_grid, "temperature_obs", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),
    "temp_mod": table_to_dataarray(
        nbfsmn_model_grid, "temperature_model", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),
    "salt_obs": table_to_dataarray(
        nbfsmn_obs_grid, "salinity_obs", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),
    "salt_mod": table_to_dataarray(
        nbfsmn_model_grid, "salinity_model", ("Site", "Depth", "Day"),
        nbfsmn_coordinates,
    ),

    # PLT nutrients
    "NO3_obs": table_to_dataarray(
        nitrogen_grid, "NO3_obs", ("Depth", "Date"),
        {"Depth": depth_values, "Date": date_values},
    ),
    "NO3_mod": table_to_dataarray(
        nitrogen_grid, "NO3_model", ("Depth", "Date"),
        {"Depth": depth_values, "Date": date_values},
    ),
    "NH4_obs": table_to_dataarray(
        nitrogen_grid, "NH4_obs", ("Depth", "Date"),
        {"Depth": depth_values, "Date": date_values},
    ),
    "NH4_mod": table_to_dataarray(
        nitrogen_grid, "NH4_model", ("Depth", "Date"),
        {"Depth": depth_values, "Date": date_values},
    ),
    "Si_obs": table_to_dataarray(
        silicate_grid, "Si_obs", ("Depth", "DateSi"),
        {"Depth": depth_values, "DateSi": date_si_values},
    ),
    "Si_mod": table_to_dataarray(
        silicate_grid, "Si_model", ("Depth", "DateSi"),
        {"Depth": depth_values, "DateSi": date_si_values},
    ),
    "SD_obs": table_to_dataarray(
        secchi_grid, "Secchi_obs", ("Week",), {"Week": week_values},
    ),
    "SD_mod": table_to_dataarray(
        secchi_grid, "Secchi_model", ("Week",), {"Week": week_values},
    ),

    # CHRP observations retain a NaN bottom layer for compatibility.
    "NO3_chrp_obs": table_to_dataarray(
        chrp_obs_grid, "CHRP_NO3_obs", ("Station", "Depth", "DateCHRP"),
        {"Station": station_values, "Depth": depth_values,
         "DateCHRP": chrp_date_values},
    ),
    "NO3_chrp_mod": table_to_dataarray(
        chrp_model_grid, "CHRP_NO3_model", ("Station", "Depth", "DateCHRP"),
        {"Station": station_values, "Depth": depth_values,
         "DateCHRP": chrp_date_values},
    ),
    "NH4_chrp_obs": table_to_dataarray(
        chrp_obs_grid, "CHRP_NH4_obs", ("Station", "Depth", "DateCHRP"),
        {"Station": station_values, "Depth": depth_values,
         "DateCHRP": chrp_date_values},
    ),
    "NH4_chrp_mod": table_to_dataarray(
        chrp_model_grid, "CHRP_NH4_model", ("Station", "Depth", "DateCHRP"),
        {"Station": station_values, "Depth": depth_values,
         "DateCHRP": chrp_date_values},
    ),
    "Si_chrp_obs": table_to_dataarray(
        chrp_obs_grid, "CHRP_Si_obs", ("Station", "Depth", "DateCHRP"),
        {"Station": station_values, "Depth": depth_values,
         "DateCHRP": chrp_date_values},
    ),
    "Si_chrp_mod": table_to_dataarray(
        chrp_model_grid, "CHRP_Si_model", ("Station", "Depth", "DateCHRP"),
        {"Station": station_values, "Depth": depth_values,
         "DateCHRP": chrp_date_values},
    ),

    # Annual production and monthly sediment fluxes
    "PP_obs": table_to_dataarray(
        production_grid, "primary_production_obs", ("Station",),
        {"Station": station_values},
    ),
    "PP_mod": table_to_dataarray(
        production_grid, "primary_production_model", ("Station",),
        {"Station": station_values},
    ),
    "sed_o2_obs": table_to_dataarray(
        sediment_obs_grid, "SOD_obs", ("Loc", "Month"),
        {"Loc": loc_values, "Month": month_values},
    ),
    "sed_o2_mod": table_to_dataarray(
        sediment_model_grid, "SOD_model", ("Loc", "Month"),
        {"Loc": loc_values, "Month": month_values},
    ),
    "sed_no3_obs": table_to_dataarray(
        sediment_obs_grid, "benthic_NO3_obs", ("Loc", "Month"),
        {"Loc": loc_values, "Month": month_values},
    ),
    "sed_no3_mod": table_to_dataarray(
        sediment_model_grid, "benthic_NO3_model", ("Loc", "Month"),
        {"Loc": loc_values, "Month": month_values},
    ),
    "sed_nh4_obs": table_to_dataarray(
        sediment_obs_grid, "benthic_NH4_obs", ("Loc", "Month"),
        {"Loc": loc_values, "Month": month_values},
    ),
    "sed_nh4_mod": table_to_dataarray(
        sediment_model_grid, "benthic_NH4_model", ("Loc", "Month"),
        {"Loc": loc_values, "Month": month_values},
    ),
}

# Preserve the exact legacy cost-variable names and dimensions.
for cost_name in [
    "NO3_cost", "NH4_cost", "pH_cost", "temperature_cost",
    "salt_cost", "Si_cost", "oxygen_cost", "NO3_chrp_cost",
    "NH4_chrp_cost", "Si_chrp_cost",
]:
    data_vars[cost_name] = xr.DataArray(
        np.full(len(depth_values), np.nan),
        dims=("Depth",),
        coords={"Depth": depth_values},
        attrs=COST_ATTRS.copy(),
    )

for cost_name in [
    "PP_cost", "SD_cost", "sed_o2_cost", "sed_no3_cost", "sed_nh4_cost",
]:
    data_vars[cost_name] = xr.DataArray(
        np.nan,
        attrs=COST_ATTRS.copy(),
    )

data_vars["param_vals"] = xr.DataArray(
    parameter_values,
    dims=("Parameters",),
    coords={"Parameters": parameter_names},
    attrs=parameter_attrs,
)

ds_test = xr.Dataset(
    data_vars=data_vars,
    coords={
        "Site": ("Site", site_values, site_attrs),
        "Depth": ("Depth", depth_values, depth_attrs),
        "Month": ("Month", month_values),
        "Day": ("Day", day_values, {
            "long_name": "local calendar day of year",
            "calendar": "proleptic_gregorian",
        }),
        "Date": ("Date", date_values, {
            "long_name": "PLT nitrogen sampling time",
            "timezone": "America/New_York",
        }),
        "DateSi": ("DateSi", date_si_values, {
            "long_name": "PLT silicate sampling time",
            "timezone": "America/New_York",
        }),
        "DateCHRP": ("DateCHRP", chrp_date_values, {
            "long_name": "CHRP sampling date",
            "timezone": "America/New_York",
        }),
        "Week": ("Week", week_values, {
            "long_name": "Secchi sampling time",
            "timezone": "America/New_York",
        }),
        "Station": ("Station", station_values, oviatt_station_attrs),
        "Loc": ("Loc", loc_values, fulweiler_location_attrs),
        "NutrientStation": (
            "NutrientStation", np.arange(0, 17), nutrient_station_attrs
        ),
        "Parameters": ("Parameters", parameter_names),
        "parameter_description": (
            "Parameters", parameter_descriptions,
            {"long_name": "model parameter description"},
        ),
        "parameter_units": (
            "Parameters", parameter_units,
            {"long_name": "model parameter units"},
        ),
    },
    attrs={
        "title": "ROMS-COSiNE Narragansett Bay cost summary",
        "description": str(runname),
        "schema_version": "2.0-legacy-compatible",
        "rundate": str(rundate),
        "year": str(runyear),
        "leap_year": "true" if is_leap_year else "false",
        "day_count": int(n_days),
        "station_file": str(station_path),
        "production_file": str(PPPATH),
        "production_file_missing": (
            "true" if production_file_missing else "false"
        ),
        "model_source_timezone": "UTC",
        "comparison_timezone": "America/New_York",
        "salinity_quality_control": (
            "finite practical salinity below 0 PSU replaced with NaN"
        ),
        "negative_salinity_values_masked": negative_salinity_count,
        "negative_salinity_counts_by_site_depth": json.dumps(
            negative_salinity_counts_by_site_depth
        ),
        "missing_model_stations": (
            ",".join(missing_model_stations)
            if missing_model_stations else "none"
        ),
        "match_coverage": (
            match_coverage.reset_index().to_json(orient="records")
        ),
        "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "processing_summary": (
            "Observations and model results aligned by named station, "
            "depth, and local sampling date/time."
        ),
    },
)

# Attach scientific metadata after assembly so helper DataArray names do
# not affect the stable legacy variable names used in the final dataset.
for variable_name, attrs in VARIABLE_ATTRS.items():
    ds_test[variable_name].attrs.update(attrs)

# Add compact match counts to relevant observed and modeled variables.
coverage_attribute_map = {
    ("NBFSMN", "temperature"): ("temp_obs", "temp_mod"),
    ("NBFSMN", "salinity"): ("salt_obs", "salt_mod"),
    ("NBFSMN", "oxygen"): ("oxygen_obs", "oxygen_mod"),
    ("NBFSMN", "pH (NBS)"): ("pH_obs", "pH_mod"),
    ("PLT", "NO3"): ("NO3_obs", "NO3_mod"),
    ("PLT", "NH4"): ("NH4_obs", "NH4_mod"),
    ("PLT", "Si"): ("Si_obs", "Si_mod"),
    ("PLT", "Secchi depth"): ("SD_obs", "SD_mod"),
    ("CHRP", "NO3"): ("NO3_chrp_obs", "NO3_chrp_mod"),
    ("CHRP", "NH4"): ("NH4_chrp_obs", "NH4_chrp_mod"),
    ("CHRP", "Si"): ("Si_chrp_obs", "Si_chrp_mod"),
    ("Oviatt", "primary production"): ("PP_obs", "PP_mod"),
    ("Fulweiler", "SOD"): ("sed_o2_obs", "sed_o2_mod"),
    ("Fulweiler", "benthic NO3"): ("sed_no3_obs", "sed_no3_mod"),
    ("Fulweiler", "benthic NH4"): ("sed_nh4_obs", "sed_nh4_mod"),
}
for coverage_key, variable_names in coverage_attribute_map.items():
    coverage = match_coverage.loc[coverage_key]
    for variable_name in variable_names:
        ds_test[variable_name].attrs.update({
            "observation_count": int(coverage["observations"]),
            "matched_count": int(coverage["matched"]),
        })

# A descriptive alias helps new readers while ds_test preserves the name
# used throughout the original notebook and downstream analysis code.
cost_summary = ds_test

ds_test

<a id="compatibility-checks"></a>
## 7.6. Compatibility and integrity checks

These assertions make schema drift fail before costs are calculated or a file is written. Reference files are inspected when available but are not required.

In [ ]:
EXPECTED_VARIABLE_DIMS = {
    "pH_obs": ("Site", "Depth", "Day"),
    "pH_mod": ("Site", "Depth", "Day"),
    "oxygen_obs": ("Site", "Depth", "Day"),
    "oxygen_mod": ("Site", "Depth", "Day"),
    "temp_obs": ("Site", "Depth", "Day"),
    "temp_mod": ("Site", "Depth", "Day"),
    "salt_obs": ("Site", "Depth", "Day"),
    "salt_mod": ("Site", "Depth", "Day"),
    "NO3_obs": ("Depth", "Date"),
    "NO3_mod": ("Depth", "Date"),
    "NH4_obs": ("Depth", "Date"),
    "NH4_mod": ("Depth", "Date"),
    "Si_obs": ("Depth", "DateSi"),
    "Si_mod": ("Depth", "DateSi"),
    "SD_obs": ("Week",),
    "SD_mod": ("Week",),
    "NO3_chrp_obs": ("Station", "Depth", "DateCHRP"),
    "NO3_chrp_mod": ("Station", "Depth", "DateCHRP"),
    "NH4_chrp_obs": ("Station", "Depth", "DateCHRP"),
    "NH4_chrp_mod": ("Station", "Depth", "DateCHRP"),
    "Si_chrp_obs": ("Station", "Depth", "DateCHRP"),
    "Si_chrp_mod": ("Station", "Depth", "DateCHRP"),
    "PP_obs": ("Station",),
    "PP_mod": ("Station",),
    "sed_o2_obs": ("Loc", "Month"),
    "sed_o2_mod": ("Loc", "Month"),
    "sed_no3_obs": ("Loc", "Month"),
    "sed_no3_mod": ("Loc", "Month"),
    "sed_nh4_obs": ("Loc", "Month"),
    "sed_nh4_mod": ("Loc", "Month"),
    "NO3_cost": ("Depth",),
    "NH4_cost": ("Depth",),
    "pH_cost": ("Depth",),
    "temperature_cost": ("Depth",),
    "salt_cost": ("Depth",),
    "Si_cost": ("Depth",),
    "oxygen_cost": ("Depth",),
    "NO3_chrp_cost": ("Depth",),
    "NH4_chrp_cost": ("Depth",),
    "Si_chrp_cost": ("Depth",),
    "PP_cost": (),
    "SD_cost": (),
    "sed_o2_cost": (),
    "sed_no3_cost": (),
    "sed_nh4_cost": (),
    "param_vals": ("Parameters",),
}

schema_rows = []
for variable_name, expected_dims in EXPECTED_VARIABLE_DIMS.items():
    actual_dims = ds_test[variable_name].dims if variable_name in ds_test else None
    schema_rows.append({
        "variable": variable_name,
        "expected_dimensions": expected_dims,
        "actual_dimensions": actual_dims,
        "valid": actual_dims == expected_dims,
    })

dataset_schema_validation = pd.DataFrame(schema_rows).set_index("variable")
if not dataset_schema_validation["valid"].all():
    invalid = dataset_schema_validation.loc[
        ~dataset_schema_validation["valid"]
    ]
    raise AssertionError(f"Cost-summary schema mismatch:\n{invalid}")

if set(ds_test.data_vars) != set(EXPECTED_VARIABLE_DIMS):
    missing = sorted(set(EXPECTED_VARIABLE_DIMS) - set(ds_test.data_vars))
    unexpected = sorted(set(ds_test.data_vars) - set(EXPECTED_VARIABLE_DIMS))
    raise AssertionError(
        f"Unexpected cost-summary variables; missing={missing}, "
        f"unexpected={unexpected}"
    )

if ds_test.sizes["Day"] != n_days:
    raise AssertionError(
        f"Day dimension should contain {n_days} days for {runyear}."
    )

for variable_name in [
    "NO3_chrp_obs", "NH4_chrp_obs", "Si_chrp_obs",
]:
    if ds_test.sizes["DateCHRP"] and not bool(
        ds_test[variable_name].sel(Depth="Bottom").isnull().all()
    ):
        raise AssertionError(
            f"{variable_name} bottom values must remain NaN."
        )

cost_variables = [
    name for name in EXPECTED_VARIABLE_DIMS
    if name.endswith("_cost")
]
if not all(bool(ds_test[name].isnull().all()) for name in cost_variables):
    raise AssertionError("Pending cost variables must be initialized with NaN.")

# Compare variable names and dimension order with known legacy files.
legacy_reference_paths = [
    COST_DIR / "LHS1_2005.nc",
    COST_DIR / "C5_2006.nc",
]
reference_rows = []
for reference_path in legacy_reference_paths:
    if not reference_path.is_file():
        continue
    with xr.open_dataset(reference_path) as reference:
        reference_rows.append({
            "reference": reference_path.name,
            "variable_names_match": (
                set(reference.data_vars) == set(ds_test.data_vars)
            ),
            "dimension_orders_match": all(
                reference[name].dims == ds_test[name].dims
                for name in EXPECTED_VARIABLE_DIMS
            ),
        })

legacy_schema_comparison = pd.DataFrame(reference_rows).set_index(
    "reference"
)
if not legacy_schema_comparison.empty and not bool(
    legacy_schema_comparison.all(axis=None)
):
    raise AssertionError(
        f"Legacy schema comparison failed:\n{legacy_schema_comparison}"
    )

dataset_validation_summary = pd.Series({
    "data_variables": len(ds_test.data_vars),
    "coordinates": len(ds_test.coords),
    "schema_checks_passed": int(dataset_schema_validation["valid"].sum()),
    "schema_checks_total": len(dataset_schema_validation),
    "legacy_references_checked": len(legacy_schema_comparison),
    "missing_model_stations": len(missing_model_stations),
    "production_file_missing": production_file_missing,
    "pending_cost_variables": len(cost_variables),
}, name="value")

display(dataset_validation_summary.to_frame())
display(legacy_schema_comparison)

print("[7/8] Cost-summary dataset constructed and validated")
print(f"  Data variables: {len(ds_test.data_vars)}")
print(f"  Coordinates:    {len(ds_test.coords)}")

<a id="calculate-and-save-costs"></a>
# 8. Calculate and save costs

This section applies one shared normalized-MSE implementation to every target. The default reproduces legacy normalization rules; an optional mode calculates every depth-dependent weight separately for surface and bottom.

<a id="cost-settings"></a>
## 8.1. Cost-component definitions

The run-level weighting and save controls are set in the top configuration cell. The table below defines each component's model variable, observation variable, weight source, legacy scope, and standard-deviation convention.

<div class="alert alert-warning">
<strong>Overwrite policy:</strong> With saving and overwriting enabled, an existing `cost_path` is replaced only after a temporary NetCDF file has been written, reopened, and validated.
</div>

In [ ]:
# Each entry describes scientific choices, not calculation mechanics.
# The shared function in the next cell handles pairing, sums of squared
# errors, invalid weights, metadata, and reporting consistently.
COST_COMPONENTS = {
    "temperature_cost": {
        "model": "temp_mod", "observation": "temp_obs",
        "weight_source": "observation", "legacy_scope": "per_depth",
        "ddof": 0,
    },
    "salt_cost": {
        "model": "salt_mod", "observation": "salt_obs",
        "weight_source": "observation", "legacy_scope": "per_depth",
        "ddof": 0,
    },
    "pH_cost": {
        "model": "pH_mod", "observation": "pH_obs",
        "weight_source": "observation", "legacy_scope": "per_depth",
        "ddof": 0,
    },
    "oxygen_cost": {
        "model": "oxygen_mod", "observation": "oxygen_obs",
        "weight_source": "observation", "legacy_scope": "per_depth",
        "ddof": 0,
    },
    "NO3_cost": {
        "model": "NO3_mod", "observation": "NO3_obs",
        "weight_source": "model", "legacy_scope": "all_depths",
        "ddof": 0,
    },
    "NH4_cost": {
        "model": "NH4_mod", "observation": "NH4_obs",
        "weight_source": "model", "legacy_scope": "all_depths",
        "ddof": 0,
    },
    "Si_cost": {
        "model": "Si_mod", "observation": "Si_obs",
        "weight_source": "model", "legacy_scope": "all_depths",
        "ddof": 0,
    },
    "NO3_chrp_cost": {
        "model": "NO3_chrp_mod", "observation": "NO3_chrp_obs",
        "weight_source": "model", "legacy_scope": "all_depths",
        "ddof": 0,
    },
    "NH4_chrp_cost": {
        "model": "NH4_chrp_mod", "observation": "NH4_chrp_obs",
        "weight_source": "model", "legacy_scope": "all_depths",
        "ddof": 0,
    },
    "Si_chrp_cost": {
        "model": "Si_chrp_mod", "observation": "Si_chrp_obs",
        "weight_source": "model", "legacy_scope": "all_depths",
        "ddof": 0,
    },
    "PP_cost": {
        "model": "PP_mod", "observation": "PP_obs",
        "weight_source": "observation", "legacy_scope": "all_values",
        "ddof": 0,
    },
    "SD_cost": {
        "model": "SD_mod", "observation": "SD_obs",
        "weight_source": "observation", "legacy_scope": "all_values",
        # pandas.Series.std() used ddof=1 in the legacy Secchi code.
        "ddof": 1,
    },
    "sed_o2_cost": {
        "model": "sed_o2_mod", "observation": "sed_o2_obs",
        "weight_source": "model", "legacy_scope": "all_values",
        "ddof": 0,
    },
    "sed_no3_cost": {
        "model": "sed_no3_mod", "observation": "sed_no3_obs",
        "weight_source": "model", "legacy_scope": "all_values",
        "ddof": 0,
    },
    "sed_nh4_cost": {
        "model": "sed_nh4_mod", "observation": "sed_nh4_obs",
        "weight_source": "model", "legacy_scope": "all_values",
        "ddof": 0,
    },
}

## 8.2. Shared cost helpers

A valid pair requires both the model result and observation to be finite. The normalization weight is calculated from the complete configured source array—not only paired samples—to preserve the legacy definition. Components with no valid pairs or an unusable weight remain `NaN` and are documented rather than silently treated as zero.

In [ ]:
COST_FORMULA = (
    "sum_squared_error / "
    "(valid_pair_count * normalization_weight^2)"
)


def finite_standard_deviation(data, ddof=0):
    """Return a standard deviation after explicitly removing NaN/inf."""
    finite_values = np.asarray(data.values, dtype=float).ravel()
    finite_values = finite_values[np.isfinite(finite_values)]

    # At least ddof + 1 values are required for a defined variance.
    if finite_values.size <= ddof:
        return np.nan
    return float(np.std(finite_values, ddof=ddof))


def calculate_normalized_mse(model, observation, weight_data, ddof=0):
    """
    Calculate one normalized mean-squared-error cost component.

    model and observation must already describe the same labeled grid.
    weight_data can cover a broader scope, such as both depths, because
    legacy PLT and CHRP nutrient costs use one combined-depth weight.
    """
    model, observation = xr.align(model, observation, join="exact")
    valid_pairs = np.isfinite(model) & np.isfinite(observation)
    valid_pair_count = int(valid_pairs.sum().item())
    normalization_weight = finite_standard_deviation(weight_data, ddof)

    if valid_pair_count == 0:
        return {
            "valid_pair_count": 0,
            "sum_squared_error": np.nan,
            "normalization_weight": normalization_weight,
            "cost": np.nan,
            "status": "no_valid_pairs",
        }

    residual = (model - observation).where(valid_pairs)
    sum_squared_error = float((residual**2).sum(skipna=True).item())

    if not np.isfinite(sum_squared_error):
        status = "invalid_sum_squared_error"
        cost = np.nan
    elif not np.isfinite(normalization_weight):
        status = "invalid_normalization_weight"
        cost = np.nan
    elif normalization_weight <= 0:
        status = "nonpositive_normalization_weight"
        cost = np.nan
    else:
        status = "calculated"
        cost = (
            sum_squared_error
            / valid_pair_count
            / normalization_weight**2
        )

    return {
        "valid_pair_count": valid_pair_count,
        "sum_squared_error": sum_squared_error,
        "normalization_weight": normalization_weight,
        "cost": float(cost),
        "status": status,
    }


def metadata_json(mapping):
    """Serialize labeled metadata with strict, NetCDF-safe JSON."""
    clean_mapping = {}
    for key, value in mapping.items():
        if isinstance(value, (np.integer, int)):
            clean_mapping[str(key)] = int(value)
        elif isinstance(value, (np.floating, float)):
            clean_mapping[str(key)] = (
                float(value) if np.isfinite(value) else None
            )
        else:
            clean_mapping[str(key)] = str(value)
    return json.dumps(clean_mapping, allow_nan=False)

<a id="calculate-costs"></a>
## 8.3. Calculate component and total costs

The displayed diagnostics are the human-readable audit trail. The same counts, sums of squared errors, weights, scopes, and statuses are attached to each cost variable before saving.

In [ ]:
cost_diagnostic_rows = []

for cost_name, specification in COST_COMPONENTS.items():
    model_name = specification["model"]
    observation_name = specification["observation"]
    model = ds_test[model_name]
    observation = ds_test[observation_name]
    weight_name = (
        model_name
        if specification["weight_source"] == "model"
        else observation_name
    )
    weight_source = ds_test[weight_name]
    ddof = specification["ddof"]
    has_depth = "Depth" in ds_test[cost_name].dims

    if has_depth:
        weight_scope = (
            specification["legacy_scope"]
            if DEPTH_WEIGHT_MODE == "legacy"
            else "per_depth"
        )

        for depth in ds_test["Depth"].values.tolist():
            depth_weight_source = (
                weight_source.sel(Depth=depth)
                if weight_scope == "per_depth"
                else weight_source
            )
            result = calculate_normalized_mse(
                model.sel(Depth=depth),
                observation.sel(Depth=depth),
                depth_weight_source,
                ddof=ddof,
            )
            ds_test[cost_name].loc[{"Depth": depth}] = result["cost"]
            cost_diagnostic_rows.append({
                "cost_variable": cost_name,
                "depth": str(depth),
                "model_variable": model_name,
                "observation_variable": observation_name,
                "weight_variable": weight_name,
                "weight_scope": weight_scope,
                "standard_deviation_ddof": ddof,
                **result,
            })
    else:
        weight_scope = specification["legacy_scope"]
        result = calculate_normalized_mse(
            model, observation, weight_source, ddof=ddof,
        )
        # Assign through .data so the existing scalar variable and its
        # metadata remain intact.
        ds_test[cost_name].data = np.asarray(result["cost"])
        cost_diagnostic_rows.append({
            "cost_variable": cost_name,
            "depth": "not_applicable",
            "model_variable": model_name,
            "observation_variable": observation_name,
            "weight_variable": weight_name,
            "weight_scope": weight_scope,
            "standard_deviation_ddof": ddof,
            **result,
        })

cost_diagnostics = pd.DataFrame(cost_diagnostic_rows)

# Attach a concise but complete audit trail to each cost variable. JSON
# is used only where metadata vary by depth because NetCDF attributes
# cannot naturally represent a labeled mapping.
for cost_name, group in cost_diagnostics.groupby(
    "cost_variable", sort=False
):
    first = group.iloc[0]
    attributes = {
        "units": "nondimensional",
        "formula": COST_FORMULA,
        "valid_pair_definition": (
            "model and observation are both finite"
        ),
        "model_variable": first["model_variable"],
        "observation_variable": first["observation_variable"],
        "normalization_variable": first["weight_variable"],
        "normalization_statistic": "standard deviation",
        "normalization_scope": first["weight_scope"],
        "standard_deviation_ddof": int(
            first["standard_deviation_ddof"]
        ),
    }

    if first["depth"] == "not_applicable":
        attributes.update({
            "valid_pair_count": int(first["valid_pair_count"]),
            "sum_squared_error": float(first["sum_squared_error"]),
            "normalization_weight": float(
                first["normalization_weight"]
            ),
            "calculation_status": first["status"],
        })
    else:
        by_depth = group.set_index("depth")
        attributes.update({
            "depth_weight_mode": DEPTH_WEIGHT_MODE,
            "valid_pair_count_by_depth": metadata_json(
                by_depth["valid_pair_count"].to_dict()
            ),
            "sum_squared_error_by_depth": metadata_json(
                by_depth["sum_squared_error"].to_dict()
            ),
            "calculation_status_by_depth": metadata_json(
                by_depth["status"].to_dict()
            ),
        })
        if first["weight_scope"] == "all_depths":
            # A combined-depth calculation has one nonredundant weight.
            attributes["normalization_weight"] = float(
                first["normalization_weight"]
            )
        else:
            attributes["normalization_weight_by_depth"] = metadata_json(
                by_depth["normalization_weight"].to_dict()
            )

    # Remove the placeholder-only metadata now that this variable has
    # a real result (or a documented reason for remaining NaN).
    ds_test[cost_name].attrs.pop("missing_value_meaning", None)
    ds_test[cost_name].attrs.pop("calculation_status", None)
    ds_test[cost_name].attrs.update(attributes)

# Build labeled lists so the total is auditable. A missing component is
# excluded rather than interpreted as a zero-cost perfect model result.
included_costs = {}
excluded_costs = []
for cost_name in COST_COMPONENTS:
    cost_data = ds_test[cost_name]
    if "Depth" in cost_data.dims:
        for depth in cost_data["Depth"].values.tolist():
            label = f"{cost_name}[{depth}]"
            value = float(cost_data.sel(Depth=depth).item())
            if np.isfinite(value):
                included_costs[label] = value
            else:
                excluded_costs.append(label)
    else:
        value = float(cost_data.item())
        if np.isfinite(value):
            included_costs[cost_name] = value
        else:
            excluded_costs.append(cost_name)

total_cost = (
    float(sum(included_costs.values()))
    if included_costs else np.nan
)
ds_test.attrs.update({
    "total_cost": total_cost,
    "total_cost_policy": "sum of finite component costs only",
    "included_cost_component_count": len(included_costs),
    "excluded_cost_component_count": len(excluded_costs),
    "included_cost_components": json.dumps(list(included_costs)),
    "excluded_cost_components": json.dumps(excluded_costs),
    "depth_weight_mode": DEPTH_WEIGHT_MODE,
    "cost_method_version": "2.0",
    "software_versions": json.dumps(SOFTWARE_VERSIONS),
})

cost_summary = ds_test

# Print one compact summary followed by the detailed, sortable table.
status_counts = cost_diagnostics["status"].value_counts()
print("[8/8] Cost calculation complete")
print(f"  Finite total cost: {total_cost:.8g}")
print(f"  Included components: {len(included_costs)}")
print(f"  Excluded components: {len(excluded_costs)}")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

diagnostic_columns = [
    "cost_variable", "depth", "valid_pair_count",
    "sum_squared_error", "normalization_weight", "cost", "status",
    "weight_variable", "weight_scope", "standard_deviation_ddof",
]
display(cost_diagnostics[diagnostic_columns].set_index(
    ["cost_variable", "depth"]
))

<a id="save-cost-summary"></a>
## 8.4. Validate and save the final cost summary

The dataset is first written beside `cost_path` under a unique temporary name. The temporary file is reopened and checked before atomically replacing the destination, so a failed write leaves an existing cost file untouched.

In [ ]:
# Confirm that dataset values and diagnostics agree before any disk I/O.
for row in cost_diagnostics.itertuples(index=False):
    stored = ds_test[row.cost_variable]
    if row.depth != "not_applicable":
        stored = stored.sel(Depth=row.depth)
    stored_value = float(stored.item())
    if not np.isclose(stored_value, row.cost, equal_nan=True):
        raise AssertionError(
            f"Cost validation failed for {row.cost_variable} "
            f"at {row.depth}."
        )

finite_dataset_costs = []
for cost_name in COST_COMPONENTS:
    values = np.asarray(ds_test[cost_name].values, dtype=float).ravel()
    finite_dataset_costs.extend(values[np.isfinite(values)].tolist())
validated_total_cost = (
    float(sum(finite_dataset_costs))
    if finite_dataset_costs else np.nan
)
if not np.isclose(validated_total_cost, total_cost, equal_nan=True):
    raise AssertionError("The stored component costs do not sum to total_cost.")

required_cost_metadata = {
    "formula", "valid_pair_definition", "model_variable",
    "observation_variable", "normalization_variable",
    "normalization_statistic", "normalization_scope",
    "standard_deviation_ddof",
}
for cost_name in COST_COMPONENTS:
    missing_attributes = (
        required_cost_metadata - set(ds_test[cost_name].attrs)
    )
    if missing_attributes:
        raise AssertionError(
            f"{cost_name} is missing metadata: "
            f"{sorted(missing_attributes)}"
        )

print("  Cost values and metadata passed pre-save validation.")

cost_file_saved = False
saved_file_size_bytes = np.nan

if SAVE_COST_FILE:
    cost_path = Path(cost_path)
    if not cost_path.parent.is_dir():
        raise FileNotFoundError(
            f"Cost output directory does not exist: {cost_path.parent}"
        )
    if cost_path.exists() and not OVERWRITE_EXISTING_COST_FILE:
        raise FileExistsError(
            f"Cost file already exists and overwrite is disabled: {cost_path}"
        )
    if cost_path.exists():
        print("WARNING: Existing cost file will be overwritten:")
        print(f"  {cost_path}")

    # NamedTemporaryFile supplies a unique path on the same filesystem,
    # which is required for the final atomic replacement.
    with tempfile.NamedTemporaryFile(
        prefix=f".{cost_path.stem}_",
        suffix=".tmp.nc",
        dir=cost_path.parent,
        delete=False,
    ) as temporary_file:
        temporary_path = Path(temporary_file.name)

    try:
        ds_test.attrs["saved_utc"] = pd.Timestamp.now(
            tz="UTC"
        ).isoformat()
        ds_test.to_netcdf(temporary_path, mode="w")

        # Reopen the file before replacement so serialization errors or
        # incomplete output cannot destroy a valid existing cost file.
        with xr.open_dataset(temporary_path) as saved_dataset:
            if set(saved_dataset.data_vars) != set(ds_test.data_vars):
                raise AssertionError(
                    "Saved file has unexpected data variables."
                )
            for variable_name, expected_dims in EXPECTED_VARIABLE_DIMS.items():
                if saved_dataset[variable_name].dims != expected_dims:
                    raise AssertionError(
                        f"Saved dimensions changed for {variable_name}."
                    )
            for cost_name in COST_COMPONENTS:
                if not np.allclose(
                    saved_dataset[cost_name].values,
                    ds_test[cost_name].values,
                    equal_nan=True,
                ):
                    raise AssertionError(
                        f"Saved values changed for {cost_name}."
                    )
            if not np.isclose(
                float(saved_dataset.attrs["total_cost"]),
                total_cost,
                equal_nan=True,
            ):
                raise AssertionError("Saved total_cost did not validate.")

        temporary_path.replace(cost_path)
        cost_file_saved = True
        saved_file_size_bytes = cost_path.stat().st_size
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    print("Cost summary saved successfully")
    print(f"  Path: {cost_path}")
    print(f"  Total finite cost: {total_cost:.8g}")
    print(f"  File size: {saved_file_size_bytes:,} bytes")
else:
    print("SAVE_COST_FILE is False; the validated dataset was not written.")

<a id="execution-summary"></a>
## 8.5. Execution summary

This final table is intentionally compact so batch logs identify the run, scientific settings, coverage, output status, and elapsed time without inspecting earlier notebook output.

In [ ]:
elapsed_seconds = time.perf_counter() - execution_started
execution_summary = pd.Series(
    {
        "run_name": matched_runname,
        "run_year": runyear,
        "station_file": str(station_path),
        "cost_path": str(cost_path),
        "depth_weight_mode": DEPTH_WEIGHT_MODE,
        "matched_observations": int(match_coverage["matched"].sum()),
        "included_cost_components": len(included_costs),
        "excluded_cost_components": len(excluded_costs),
        "total_finite_cost": total_cost,
        "cost_file_saved": cost_file_saved,
        "saved_file_size_bytes": saved_file_size_bytes,
        "elapsed_seconds": round(elapsed_seconds, 2),
    },
    name="value",
)

print("SUCCESS: Cost-function workflow completed")
display(execution_summary.to_frame())

# Close the source handle last. All output arrays and validation tables
# remain available in memory for inspection after interactive execution.
sta.close()